# DwTA POC — Specification v1

**Dwell-Window Threat Assurance (DwTA): lab, rules-first detection core and dwell-window incident review, built locally on a Mac**

| Field | Value |
|---|---|
| Version | **1.0** (first major version) |
| Date | 2026-11-03 |
| Status | Baseline for build. Changes after this point are tracked as v1.x revisions or v2 |
| Parent document | `DwTA-POC-Implementation-Plan-v0.2.md` |
| Merged inputs | `SPEC-read-budget-av-integration.md` (`RB-` IDs, §5.11) |
| Building blocks | **B1** SMB client · **B2** Samba share · **B3** Attack simulator · **B4** Collector · **B5** Classifier · **B6** Ledger · **B7** Threat feed · **B8** Feed matcher · **B9** Simulated incident · **B10** Incident engine |
| Target platform | The team's reference Mac (§3): MacBook Pro M4 Pro, macOS 26.1, Docker Desktop 29.9 |
| Review history | The working draft `DwTA-Spec-v1-Foundation.ipynb` keeps the full review trail |

### Summary
**Core idea.** When a ransomware incident is detected, the Nasuni admin gives DwTA a **restore time**. **Before restoring**, DwTA scores the files written since then, **rescans the suspicious ones with antivirus using today's signatures**, and shows them, each with a **suspicion score and level**, the reasons and the antivirus verdict. The admin marks each one **remove** or **keep**, and the restore runs **without the removed files**. Dormant files the attacker planted therefore don't survive the recovery to be reused in a later attack.

What makes that possible, all recorded at write time:
1. **Day 1:** classify every written file **by content** (rules, plus Magika as the AI layer), hash it, and remember it in a ledger with who wrote it and from where.
2. **Day 5:** when threat intelligence later lists the file's hash, find it by **lookup**, never by rescanning.
3. **Day 7:** on the incident, score every file written between the restore time and detection, rescan the flagged ones with antivirus, list only the suspicious ones, record the admin's decisions and hand a **restore-exclusion manifest** to the restore step.

v1 builds all of this in an isolated Docker Compose lab on one Mac, driven by a safe, scenario-based attack simulator on a simulated clock. In the product, DwTA adds **no file reads** beyond the existing antivirus pass (§5.11).

### How to read and use this notebook
- **Core idea**: the pre-restore review flow, step by step.
- §1–4: scope, topology, target machine, how users are simulated.
- §5: the ten building blocks and the cross-cutting read budget, each with numbered requirements (`MUST` = required for v1, `SHOULD` = if time permits).
- §6: data contracts, the part the product would adopt (including the restore-exclusion manifest). §7: acceptance tests. §8: layout and configuration.
- §9: open questions and the decision log. §10: backlog. §11: work plan.
- Appendix A: run **Cell P1 → P2 → P3** to check that the Mac matches §3 and to see this instance's address plan. **P4** (pull images, write `.env`) runs only when `RUN_SETUP = True`.

## Core idea: a pre-restore review that keeps dormant attacker files out of the recovery

**The problem.** In a ransomware attack, the attacker often plants files (payloads, tools, disguised executables) days before encrypting anything, and leaves them dormant. When the admin recovers with Targeted Restore, the encrypted files come back, but those planted files are still on the share. The attacker can reuse them for the next attack.

**What DwTA gives the admin.** Intelligence about which files written since the chosen restore point look suspicious, **before** the restore, so the admin can take them out of the recovery.

```
 ① Ransomware incident detected        (product: detection time, user, IP, machine)
            │
 ② Admin enters the RESTORE TIME        (latest moment the admin believes had no attacker presence)
            │
 ③ DwTA reviews every file written between the restore time and detection, using what it
   recorded at write time: content type (magic bytes / Magika), hash, macro and YARA results,
   extension mismatch, writer, IP, machine, version history, threat-feed matches
            │
 ④ DwTA scores each file and keeps only the SUSPICIOUS ones (suspicion score ≥ threshold)
            │
 ⑤ DwTA RESCANS the flagged files with antivirus using TODAY's signatures
   (a file that passed antivirus on Day 1 may be detected now; detection adds to its score)
            │
 ⑥ DwTA shows the suspicious files, ranked by suspicion score with a level
   (high / medium / low), the reasons behind each score and the antivirus verdict
            │
 ⑦ Admin marks each file REMOVE or KEEP, then finalizes
            │
 ⑧ Restore runs to the restore time WITHOUT the removed files
   (removed files go to a locked quarantine folder: never deleted, reversible)
```

**Why it works cheaply.** Steps ③–④ read **no file content**: everything comes from the ledger, filled at write time (§5.5) from a single read per version, which in the product is the antivirus pass (§5.11). Step ⑤ is the only content read after write time, and it reads **only the few flagged files**, never the share (§5.10.4).

**How v1 realizes each step:**
| Step | v1 (POC) | Block |
|---|---|---|
| ① Incident | Simulated by the scenario or CLI (stand-in for Ransomware Protection) | B9 |
| ② Restore time | `restore_time` in the scenario, or `dwta-incident set-restore-time` | B9 |
| ③ Review | Window query over the ledger, attack-outputs grouping | B5, B6, B10 |
| ④ Scoring | Configurable signals and threshold (`ranking.toml`) | B10 |
| ⑤ AV rescan of flagged files | **Spec now, built in v2** with a ClamAV container (D-37). v1 marks every item `av_verdict = not_scanned` through the same interface | B10 (+ ClamAV in v2) |
| ⑥ Suspicious list | Ranked **review list** with `suspicion_score`, `suspicion_level`, cited reasons and the AV verdict | B10 |
| ⑦ Decisions | `dwta-incident decide … remove|keep`, then `dwta-incident finalize` → **restore-exclusion manifest** (§6.7) | B10 |
| ⑧ Restore without removed files | **Later (API/UI, BL-06):** Targeted Restore + move removed files to a locked quarantine folder. v1 delivers the manifest that drives it | — |

## 1. Scope

### 1.1 In scope (v1)
Build these ten blocks **from scratch**, in an isolated Docker Compose lab on one Mac. B1–B4 are the lab foundation (event capture). B5–B10 are the detection core (classify → remember → re-check → rank):

| ID | Block | One-line purpose |
|---|---|---|
| B1 | **SMB client** | Writes files to the lab share as distinct, attributable identities: a container per identity, plus a native macOS mount |
| B2 | **Samba share** | Serves `finance` and `software` shares and audits every file operation with user, client IP, machine name, share, operation and path (`vfs_full_audit`) |
| B3 | **Attack simulator** (`dwta-sim`) | Generic, safe, scenario-driven threat simulator. It runs any attack written as a TOML scenario file by remote-controlling the client containers, and records an answer key (ground truth) of what it did. **v1 implements one scenario: `dwell-window`**; others are added as needed |
| B4 | **Collector** | Tails the Samba audit log asynchronously and turns raw operations into one clean file event per completed write, rename or delete. Follows temp-file-then-rename writes |
| B5 | **Classifier interface** | Reads each written file from disk (read-only), hashes it (SHA-256), types it by content (rules: magic bytes; AI: Magika), runs oletools and YARA-X, flags extension mismatch, raises write-time alerts (Day 1) |
| B6 | **Artifact ledger** | **Postgres 16** database: one row per file version with type, hash, writer and IP, plus feed entries, matches, alerts, incidents and review lists. **The schema is the product contract** |
| B7 | **Threat feed** | Fabricated feed in MalwareBazaar CSV format, published by the simulator (`publish_feed`, Day 5) into a feed inbox |
| B8 | **Feed matcher** | Ingests feeds and looks the hashes up in the ledger. **No rescan, no file reads.** Raises a critical retroactive alert (Day 5) |
| B9 | **Simulated incident** | Stand-in for the product's Ransomware Protection alert (detection time, user, IP, machine, volume) plus the admin's **restore time**. Raised by the scenario (Day 7) or from the CLI |
| B10 | **Incident engine** | **Pre-restore review** (see Core idea): looks at every file written between the admin's **restore time** and **detection**, scores each with configurable rules (suspicion score + level), **rescans the flagged files with antivirus** (v2; v1 marks them `not_scanned`), lists only the suspicious ones, records the admin's **remove** / **keep** decisions and produces the **restore-exclusion manifest**. For both rules and rules + AI |

### 1.2 Out of scope (v1)
- Narrator (template/LLM), API and UI (later spec versions).
- **Executing** the restore with exclusions (Targeted Restore + moving removed files to a locked quarantine folder): comes with API/UI (D-17). v1 produces the review list, records decisions and writes the restore-exclusion manifest.
- A full list of everything Targeted Restore will restore: not our job (D-23). DwTA lists only the files that need human input.
- TLSH similarity, entropy and pefile heuristics, EMBER: v2 (BL-10).
- Real MalwareBazaar feed and a standalone feed service: later (BL-09, BL-11).
- **ClamAV container** (Day-1 baseline and the step ⑤ rescan): v2 (D-37); see §8.3.
- **UNTY-89818**: not used. The attack simulator is a new standalone script (B3).
- Windows VM client: not planned (D-13). Office/Explorer save behavior is emulated with office-style writes (§5.3.2).
- Any production Edge, NMC or Ransomware Protection code.

### 1.3 Guiding constraints inherited from the plan
- **Isolated:** everything runs in its own Compose project and network, and needs no internet once prerequisites are pulled.
- **Never in the write path:** the collector reads logs after the fact. Samba never waits on it.
- **Safe by construction:** no real malware. Only benign generated files, EICAR and InfoSec-approved test artifacts.
- **Multi-instance friendly:** several labs, or other SMB servers, may run on the same Mac at once. Nothing may hard-code port 445, a fixed subnet or a fixed container name.

## 2. Lab topology

### 2.1 The picture: every container, end to end
```
 macOS host                             Docker Desktop VM (linux/arm64) · lab network <INSTANCE>-lab, e.g. 10.250.1.0/24
 ────────────────────────               ─────────────────────────────────────────────────────────────────────────────────

 dwta-sim (B3) ──docker exec──▶ B1 clients ───────────SMB3───────────▶ B2 samba (.10)
  │  │  │  │                    client-admin .5  client-jdoe .21           smbd + vfs_full_audit
  │  │  │  │                    client-asmith .22  client-attack .66          │ files       │ audit lines
  │  │  │  │   Finder / mount_smbfs ── 127.0.0.1:<SMB_PORT> ──────────────▶  ▼             ▼
  │  │  │  │                                                         data/<share>    logs/audit.log
  │  │  │  └─ simclock.jsonl ─────────────────────────────────┐          │ (ro)          │ (ro)
  │  │  │                                                     ▼          │               ▼
  │  │  │                                              ┌───────────────────────────────────────┐
  │  │  │                                              │ B4 collector (.30)                    │  CAPTURE
  │  │  │                                              └───────────────────┬───────────────────┘
  │  │  │                                                  run/journal/    │ fs_event (JSONL)
  │  │  │                                                                  ▼
  │  │  │                                   data/ (ro) ──▶ ┌───────────────────────────────────────┐
  │  │  │                                                  │ B5 classifier (.31)                   │  CLASSIFY  (Day 1)
  │  │  │                                                  │ content pass · rules + Magika ·       │
  │  │  │                                                  │ oletools · YARA-X · versions · alerts │
  │  │  │                                                  └───────────────────┬───────────────────┘
  │  │  │                                                         artifacts,   │ write-time alerts
  │  │  │                                                                      ▼
  │  │  │  D5 publish_feed                                  ┌───────────────────────────────────────┐
  │  │  └──▶ feeds/inbox ──▶ B8 matcher (.32) ─matches──────▶│ B6 ledger: Postgres 16 (.20)          │  REMEMBER /
  │  │                       (no data mount: can't rescan)   │ files · artifacts · feeds · matches · │  RE-CHECK (Day 5)
  │  │                                                       │ incidents · review · decisions ·alerts│
  │  │  D7 raise_incident                                    └───────────────────┬───────────────────┘
  │  └──▶ incidents/inbox ──▶ ┌──────────────────────────────────┐  reads/writes │
  │       (or dwta-incident)  │ B9/B10 incident (.33)            │◀──────────────┘                     REVIEW    (Day 7)
  │                           │ intake · window · scoring ·      │
  │                           │ step ⑤ AV rescan ┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┼┄┄▶ clamav (.40, v2) ┄┄ reads flagged files only
  │                           │ review list · decisions · manifest│
  │                           └───────────────┬──────────────────┘
  ▼                                           ▼
 run/runs/<id>/groundtruth.jsonl     run/incidents/<id>/review-list.json · restore-exclusions.json
```
`data/`, `logs/`, `run/`, `feeds/` and `incidents/` are folders on the Mac under `lab/<INSTANCE>/`, bind-mounted into the containers; `(ro)` = read-only. The ledger's data lives on a Docker named volume.

**Where each block lives**
| Block | Container | IP (host octet) | Reads | Writes |
|---|---|---|---|---|
| B1 SMB client | `client-admin`, `client-jdoe`, `client-asmith`, `client-attack` | .5 / .21 / .22 / .66 | staging files, corpus pack | SMB to Samba |
| B2 Samba share | `samba` | .10 | — | share files (`data/`), `logs/audit.log` |
| B3 Attack simulator | none: host CLI `dwta-sim` | — | scenario TOML | `simclock.jsonl`, ground truth, `feeds/inbox`, `incidents/inbox`; drives B1 |
| B4 Collector | `collector` | .30 | `audit.log` (ro), `data/` (ro, watcher) | `run/journal/` |
| **B5 Classifier** | `classifier` | **.31** | `run/journal/`, `data/` (ro: the single content pass) | ledger `files`, `artifacts`, write-time `alerts` |
| B6 Ledger | `ledger` (Postgres 16) | .20 | — | — (named volume) |
| B7 Threat feed | none: `publish_feed` step writes `feeds/inbox/` | — | — | feed CSV |
| B8 Feed matcher | `matcher` | .32 | `feeds/inbox/`, ledger | ledger `feed_entries`, `feed_matches`, alerts |
| B9 Simulated incident | none: `raise_incident` step or `dwta-incident raise` | — | — | `incidents/inbox/` |
| **B10 Incident engine** | `incident` | **.33** | `incidents/inbox/`, ledger; v2: asks ClamAV to rescan flagged files | ledger `incidents`, `review_items`, `review_decisions`, alerts; review list + manifest files |
| AV rescan (v2) | `clamav` | .40 | flagged files only (ro) | verdicts back to B10 |

### 2.2 How a threat is simulated, step by step
The simulator (B3) never speaks SMB itself. It is a **puppeteer**: it reads a scenario and, for each step, tells one client container to run an `smbclient` command. Example: plant a disguised file.

1. The scenario step says: *at simulated Day 1 09:14, actor `client-attack`, logged in as `svc-backup`, upload `benign_pe.bin` to `finance/Q3/Q3-Report.pdf`*.
2. `dwta-sim` writes `simclock.json` = "it is now Day 1 09:14".
3. `dwta-sim` runs `docker compose exec client-attack smbclient //samba/finance -U svc-backup … -c "put …"`.
4. The SMB connection really comes from **10.250.1.66** with account **svc-backup** and machine name **SRV-BKP01**. Samba stores the file, and `full_audit` writes lines like `…|svc-backup|10.250.1.66|SRV-BKP01|finance|pwrite|ok|Q3/Q3-Report.pdf`.
5. The collector tails that line, stamps the simulated time from `simclock.json`, and emits one `write_close` event.
6. `dwta-sim` appends what it *intended* to happen to `groundtruth.jsonl`. Tests later compare that answer key with what the collector produced.

So the "attack" is real SMB traffic from a distinct machine, account and IP. Only the payloads are harmless and the dates are simulated.

### 2.3 Address plan: what "host octet" means
An IP like `10.250.1.66` has four numbers (octets). The **subnet** (`10.250.1`) is chosen per lab instance so it avoids the Docker networks already in use on the Mac. The **host octet** (`.66`) is the last number, and it is **fixed per container**: in every instance, the attacker container is `.66` and Samba is `.10`. A second lab (`dwta2`) would get `10.250.2.66`, `10.250.2.10` and so on. Compose assigns these static addresses (`ipv4_address`), so the same container always has the same IP, and IP-based attribution is stable and repeatable.

| Container | Host octet | Default credentials | Role |
|---|---|---|---|
| `samba` | `.10` | — | SMB server (B2) |
| `collector` | `.30` | — | Audit tailer (B4) |
| `client-admin` | `.5` | `admin` (`WS-ADMIN01`) | Seeds data, admin operations |
| `client-jdoe` | `.21` | `jdoe` (`WS-JDOE01`) | Legitimate finance user's workstation |
| `client-asmith` | `.22` | `asmith` (`WS-ASMITH01`) | Legitimate user's workstation |
| `client-attack` | `.66` | `svc-backup` (`SRV-BKP01`) | **Attacker's machine**. It can log in as *any* identity whose credentials the scenario gives it (U8) |
| macOS host | Docker gateway / VM NAT IP | any | Native client. IP attribution is *not* distinctive (see §3.2) |
| `ledger` | `.20` | — | B6: Postgres 16 (no host port by default) |
| `classifier` | `.31` | — | B5 |
| `matcher` | `.32` | — | B8 (no share data mounted at all) |
| `incident` | `.33` | — | B9/B10 |
| `clamav` | `.40` | — | v2: Day-1 baseline + step ⑤ rescan (§8.3) |

### 2.4 Detection pipeline (B5–B10)
```
 collector ──▶ run/journal/*.jsonl ──▶ classifier (B5) ── reads file bytes ◀── data/ (read-only)
                (fs_event)                  │  SHA-256, type, ext mismatch, oletools, YARA-X
                                            ▼
 dwta-sim D5 publish_feed ──▶ feeds/inbox/*.csv ──▶ matcher (B8) ──▶ ┌──────────────────────────────┐
                                                    (hash lookup)    │ ledger (B6, Postgres 16)     │
 dwta-sim D7 raise_incident ─┐                                       │ artifacts · feed_entries ·   │
 dwta-incident raise (CLI) ──┴─▶ incidents/inbox/*.json ──▶ incident │ feed_matches · alerts ·      │
                                                           engine    │ incidents · review_items     │
                                                           (B10) ◀──▶└──────────────────────────────┘
                                                              │                  │
                                         run/incidents/<id>/review-list.json    run/alerts.jsonl
```
| Sim day | What happens | Block | Alert |
|---|---|---|---|
| D1 | svc-backup plants `Q3-Report.pdf` (really a PE). Classifier: `pe_binary`, extension mismatch, executable on a document share | B5 → B6 | `write_time`, **high** |
| D5 | Fabricated feed lists that SHA-256. Matcher finds it in the ledger by lookup | B7 → B8 → B6 | `feed_match`, **critical** |
| D7 | Encryption run; product detects it (time, user, IP, machine); admin gives the **restore time** (D1 00:00). Engine reviews [restore time → detection] **before the restore**, rescans the flagged files with AV (v2), and `Q3-Report.pdf` ranks #1 (**high**) and the encrypted copies + ransom note collapse into one group; admin marks them **remove** → exclusion manifest | B9 → B10 → B6 | `incident`, **critical** |

## 3. Target environment: the reference Mac

The lab is built **for this machine as it is**. Nothing extra is installed on the Mac. Everything the lab needs beyond what is listed here runs inside containers. Values were read from the machine on 2026-11-03 (`system_profiler`, `docker info`, `sw_vers`).

### 3.1 Machine profile and the design constraints it implies
| Item | Reference Mac | Constraint on what we build |
|---|---|---|
| Hardware | MacBook Pro, **Apple M4 Pro**, 14 cores (10P + 4E), 24 GB RAM | All images MUST be `linux/arm64`; no amd64-only images, no Rosetta emulation |
| OS | **macOS 26.1** (25B78) | Native client = built-in `mount_smbfs` / Finder |
| Container runtime | **Docker Desktop**, Engine 29.9.1, Compose v5.6.0, LinuxKit kernel 7.0.14 | Compose file uses only Compose-spec features supported by v5; inotify works inside the VM |
| Docker VM resources | **14 CPU, about 7.7 GB RAM** | The whole v1 lab (B1–B10, including Magika and Postgres) MUST fit in **≤ 2 GB** RAM, leaving room for other stacks and later blocks |
| Host Python | **3.14.0** at `/usr/local/bin/python3`, no extra packages | `dwta-sim` and this notebook MUST be **stdlib-only** and run on 3.14. Container Python can differ (`python:3.12-slim`) |
| Homebrew | Present (`/opt/homebrew`) | Not required by the lab |
| Ollama | **0.22.0** via Homebrew, **native** (not Docker). Model present: `llama3.1` (4.6 GB). Not running as a service today | Not used by v1 blocks. Later LLM blocks call it natively: start with `ollama serve` or `brew services start ollama`, and containers reach it at `http://host.docker.internal:11434`. Running it natively uses the Apple GPU (Metal) and keeps it out of the 7.7 GB Docker VM |
| Disk | About **30 GB free (93 % used)** | v1 images + data MUST stay under 3 GB (Magika adds ≈ 0.3 GB). Clean up before later LLM blocks |
| Host port 445 | Free today, but macOS File Sharing may take it | Never bind 445; publish `127.0.0.1:<SMB_PORT>` from 14450–14499 |
| Ports already published by other stacks | 3000, 5432, 6379, 8001, 8002, 8888 | Lab publishes **only** the SMB port |
| Docker subnets already in use | 172.17–172.31.0.0/16 (14 networks) | Lab subnets come from `10.250.1–19.0/24` |

**Before the build day (not prerequisites, just preparation):** build the service images once while online (pip wheels for `magika`, `oletools`, `yara-x` baked in), pull the base images `debian:bookworm-slim`, `alpine:3.20`, `python:3.12-slim`, `postgres:16-alpine` (Cell P4), and get InfoSec approval of the safe artifact list (§5.3.6).

### 3.2 Mac-specific gotchas the spec must handle
1. **Port 445 belongs to macOS.** When File Sharing is on, macOS's own `smbd` owns 445. The lab never binds 445; it publishes `127.0.0.1:<SMB_PORT>` only.
2. **Source IP from the Mac host is not distinctive.** Docker Desktop NATs host→container traffic, so Samba sees a gateway/VM address, not the Mac's LAN IP. Per-identity IP attribution only works for container clients.
3. **macOS clients write metadata files** (`.DS_Store`, `._*` AppleDouble). Samba uses `vfs_fruit` + `streams_xattr` to absorb most of them, and the collector ignores the rest (§5.4.5).
4. **Bind-mount file events.** inotify fires for writes made *inside* the Linux VM. The collector therefore runs as a container sharing the same volume, not on the host.

## 4. Simulating users: the attribution dimensions

The demo depends on telling the **attacker** apart from **legitimate** users. "User" is not one property. These are the dimensions we can simulate, how v1 simulates each, and whether the collector captures it.

**Key design choice: actor ≠ credentials.** A scenario step names the **actor** (which client container, so which IP and machine) *and* the **credentials** it logs in with, separately. Normally `client-jdoe` logs in as `jdoe`. But the attacker machine `client-attack` can log in as `jdoe` with stolen credentials. Without this split, U8 cannot be simulated.

| # | Dimension | How v1 simulates it | Captured by B2/B4 | Notes |
|---|---|---|---|---|
| U1 | **SMB identity** (account name) | Separate Samba users: `jdoe`, `asmith`, `svc-backup`, `admin` | Yes: `%u` in audit prefix | Primary attribution |
| U2 | **Source IP** | One client container per machine, each with a **static IP** (§2.3) | Yes: `%I` | Container clients only (§3.2) |
| U3 | **Machine / NetBIOS name** | `smbclient -n WS-JDOE01`; the attacker machine uses `SRV-BKP01` | Yes: `%m` | Cheap extra signal, easy to spoof |
| U4 | **Client type / OS** | `smbclient` (Linux) vs macOS native mount | Partly: dialect and behavior differ | No Windows client (D-13) |
| U5 | **Share and permission scope** | Groups: `finance-rw` (jdoe, asmith, svc-backup, admin), `software-rw` (asmith, admin). svc-backup having write access to finance is the misconfiguration the attacker abuses | Share name: `%S` | |
| U6 | **Behavior pattern** | Scenario steps set it: what, how much, how fast, when (rate, bulk renames, extension changes, deletes) | Raw material only: every op with time, user, IP, size, old and new path | **No behavior rules in v1.** Detection approaches: §4.1 |
| U7 | **Time of activity** | Simulated clock (§5.3.4): steps carry a simulated date/time | `sim_ts` stamped by the collector | Real wall-clock `ts` kept as well |
| U8 | **Compromised legitimate account** | Scenario step `actor: client-attack, as: jdoe` (stolen credentials) | Yes. The audit line shows `user=jdoe` but `ip=10.250.1.66`, `machine=SRV-BKP01` | Detectable because the **user (U1) and the machine/IP (U2/U3) don't match jdoe's usual pairing** (`jdoe` ↔ `.21`/`WS-JDOE01`). Catalog scenario `compromised-account` (§5.3.3, not in v1) |

**v1 decision:** identities, shares and IPs are a **fixed** table in v1 (§2.3, §5.2.3). Parameterizing them is backlog item **BL-01** for v2.

### 4.1 Behavior-pattern detection: candidate approaches (not in v1; BL-07)
v1 only guarantees the collector captures the fields these need. Options, from cheapest to richest:
1. **Threshold rules over sliding windows**, per user and per IP: writes, renames or deletes per minute; share of renames that change the extension; ransom-note-like file names. Deterministic, fits the "rules first" principle.
2. **Content signals at write time:** entropy jump between the old and new version of a file (encryption), or magic bytes that no longer match the extension.
3. **Per-identity baseline:** learn each user's normal IPs, machines, shares, hours and volume from simulated "normal days", then flag deviations (z-score). This is what makes U8 detectable automatically.
4. **Sequence / ML (stretch):** an anomaly model (for example an isolation forest) over per-session feature vectors, or an LLM that only *explains* a rule hit. It never decides.

## 5. Building-block specifications

Requirement IDs: `SMBC-` (B1), `SMBS-` (B2), `SIM-` (B3), `COL-` (B4). **MUST** is required for v1; **SHOULD** is required if time permits.

### 5.1 B1 — SMB client

**Purpose.** Produce realistic, attributable SMB writes to the lab share from (a) per-identity Linux containers and (b) the Mac itself.

#### 5.1.1 Container clients
| ID | Requirement |
|---|---|
| SMBC-01 | MUST provide one image `dwta-smbclient` (base `alpine:3.20`; packages `samba-client`, `openssl`, `bash`, `coreutils`). |
| SMBC-02 | MUST run one long-lived container per actor machine (`client-admin`, `client-jdoe`, `client-asmith`, `client-attack`), each with the static IP from §2.3, idling (`sleep infinity`) until commands are `exec`'d. |
| SMBC-03 | Credentials MUST come from auth files mounted read-only (`/run/secrets/smb-<identity>`), generated from `lab/<INSTANCE>/.env`. Each container gets its own identity's file. `client-attack` additionally gets the files that scenarios declare as "stolen" (U8). Passwords never appear in command lines, logs or git. |
| SMBC-04 | Every invocation MUST set the **actor machine's** NetBIOS name (`-n WS-JDOE01`, `WS-ASMITH01`, `SRV-BKP01`, `WS-ADMIN01`), regardless of which credentials are used, and use `--option='client min protocol=SMB2'`. |
| SMBC-05 | MUST support: list shares, `put`, `get`, `rename`, `del`, `mkdir`, and recursive copy of a directory (`mput` with `recurse`). |
| SMBC-06 | MUST NOT be reachable from outside the lab network (no published ports). |

Reference invocation (spec only, not part of the contract):
```bash
docker compose -p $INSTANCE exec client-attack \
  smbclient //samba/finance -A /run/secrets/smb-svc-backup -n SRV-BKP01 \
  -m SMB3 -c 'cd Q3; put /artifacts/benign_pe.bin Q3-Report.pdf'
```

#### 5.1.2 macOS native client
| ID | Requirement |
|---|---|
| SMBC-10 | MUST be able to mount any lab share from the Mac host at `smb://<identity>@127.0.0.1:<SMB_PORT>/<share>`, both from Finder (⌘K) and with `mount_smbfs //<identity>@127.0.0.1:<SMB_PORT>/<share> ~/dwta-mnt/<INSTANCE>/<share>`. |
| SMBC-11 | Host-originated events are attributed by **user only**; the collector records `client_ip` as observed (gateway address) and sets `client_origin = "host"` (§5.4.4). |
| SMBC-12 | SHOULD document how to unmount cleanly (`umount ~/dwta-mnt/...`), so stale mounts don't break `reset`. |

**Acceptance:** AT-03, AT-04 (§7).

---

### 5.2 B2 — Samba share

**Purpose.** An isolated SMB server whose audit log is the primary event source.

#### 5.2.1 Image and runtime
| ID | Requirement |
|---|---|
| SMBS-01 | MUST build image `dwta-samba` from `debian:bookworm-slim` with `samba` (+ `samba-vfs-modules` if `full_audit.so`/`fruit.so` are not in the base package). Image MUST be `linux/arm64`. |
| SMBS-02 | MUST create the Unix and Samba users and groups from §5.2.3 at container start (entrypoint), reading passwords from env/secrets. The operation MUST be idempotent across restarts. |
| SMBS-03 | MUST listen on 445 **inside** the container. Compose publishes it as `127.0.0.1:${SMB_PORT}:445` (never `0.0.0.0`, never host 445). |
| SMBS-04 | MUST set `server min protocol = SMB2`, disable NetBIOS/nmbd (`disable netbios = yes`, `smb ports = 445`), and refuse guest access. |
| SMBS-05 | Share data MUST be bind-mounted from `lab/<INSTANCE>/data/<share>` and the audit log from `lab/<INSTANCE>/logs/`, so they survive container restarts and `reset` can be scripted. |
| SMBS-06 | Healthcheck: `smbclient -L localhost -N` or `smbstatus` returns 0 within 30 s of start. |

#### 5.2.2 Audit configuration (`vfs_full_audit`)

**What it is and who does it.** Yes: the Samba server (the `smbd` process in the `samba` container) serves the shares to every client. Every time a client opens, writes, renames or deletes a file, `smbd` performs that operation on the Linux disk through a stack of plug-in modules called **VFS (Virtual File System) modules**. `vfs_full_audit` is one of those modules, shipped with Samba. It sits in that stack, lets each call pass through unchanged, and writes **one log line per call**, with a configurable prefix (who, from where, which share) plus the operation, result and path. It doesn't block or modify anything. It only records.

**Where the logs go.** By default `full_audit` sends lines to **syslog** (facility `USER`, priority `NOTICE`). A container has no syslog daemon, so we choose a destination explicitly (SMBS-13). Either way, the file ends up at `/var/log/samba/audit.log` *inside* the container, which is bind-mounted to **`dwta-poc/lab/<INSTANCE>/logs/audit.log` on the Mac**. The collector reads that same file read-only.

Example line (prefix `%T|%u|%I|%m|%S`, then op, result, args):
```
2026/11/03 06:12:44.381|svc-backup|10.250.1.66|SRV-BKP01|finance|renameat|ok|Q3/~WRL0001.tmp|Q3/Q3-Report.pdf
```

| ID | Requirement |
|---|---|
| SMBS-10 | Every share MUST load `vfs objects = fruit streams_xattr full_audit`. |
| SMBS-11 | Audit prefix MUST be `full_audit:prefix = %T\|%u\|%I\|%m\|%S`: time, user, client IP, client NetBIOS name, share. |
| SMBS-12 | Audited success ops MUST include at least: `connect disconnect create_file openat close pwrite pwrite_send renameat unlinkat mkdirat`. Failures: `connect create_file renameat unlinkat`. Read ops (`pread*`) are **not** audited in v1 (volume). |
| SMBS-13 | Audit records MUST land in a single file `lab/<INSTANCE>/logs/audit.log`. **Preferred:** `full_audit:syslog = false`, with the smbd log routed to that file. **Fallback:** an in-container `rsyslogd` with `local5.* /var/log/samba/audit.log` and `full_audit:facility = LOCAL5`. Which one works on Bookworm's Samba version is verification item **V-01**. |
| SMBS-14 | Log rotation: rotate at 50 MB with copy-truncate *or* rename. Either way the collector MUST survive it (COL-07). |

#### 5.2.3 Shares and identities (fixed in v1, parameterized in v2: BL-01)
| Share | Purpose in the story | Path | Write groups |
|---|---|---|---|
| `finance` | **Document share**: where the disguised file is planted and where encryption happens | `/srv/shares/finance` | `finance-rw` |
| `software` | **Software share**: executables are normal here (later false-positive comparison) | `/srv/shares/software` | `software-rw` |

| Identity | Groups | Role |
|---|---|---|
| `jdoe` | finance-rw | Legitimate |
| `asmith` | finance-rw, software-rw | Legitimate |
| `svc-backup` | finance-rw | **Attacker** (look-alike service account) |
| `admin` | finance-rw, software-rw | Seeding and admin |

**Acceptance:** AT-02, AT-03, AT-05, AT-06.

---

### 5.3 B3 — Attack simulator (`dwta-sim`): generic threat simulator

**In one paragraph.** `dwta-sim` is the threat simulator. It is a small Python program on the Mac that reads a **scenario file** (a list of steps: *who* does *what* to *which files*, at *what simulated time*). It executes each step by telling one of the client containers to run an SMB command against the lab share. Because the commands really run from those containers, Samba records real user, IP and machine attribution. The payloads are harmless and the timestamps are simulated. For every step it also writes down what it did (the ground truth), so the collector, and later the detectors, can be scored. The week-long dwell-window story is just **one scenario**, and the only one v1 builds. Other attacks become other scenario files built from the same action library, added when needed.

#### 5.3.1 Concepts
| Term | Meaning |
|---|---|
| **Scenario** | A TOML file (`scenarios/<name>.toml`, parsed with stdlib `tomllib`) with metadata, the actors used, and an ordered list of steps |
| **Step** | `at` (simulated time) + `actor` (client container) + `as` (credentials, default = actor's own) + `action` + parameters |
| **Action** | A reusable primitive from the action library (§5.3.2). New attack types should need **new scenarios, not new code**, as long as they are built from existing actions |
| **Actor** | A client container (= machine + IP), e.g. `client-attack`. Actors are the machines; credentials are separate (§4) |
| **Run** | One execution of a scenario, with a `run_id`. All output for it goes to `run/<run_id>/` |

#### 5.3.2 Action library
v1 implements only the actions that `dwell-window` needs. The rest are designed (same plug-in interface, SIM-06) and get built when a scenario needs them.

| Action | What it does over SMB | v1 |
|---|---|---|
| `seed` | Upload a corpus pack (§5.3.5) to the shares | **Yes** |
| `edit` | Legitimate-looking change to existing documents: download, modify deterministically, upload | **Yes** |
| `write_office_style` | Option of `edit`/`write` (`style = "office"`): upload `~WRL<n>.tmp`, then rename it over the target | **Yes** |
| `write` | Upload a generated or artifact file to a path | **Yes** |
| `plant` | `write` with an approved safe artifact under a disguised name (e.g. PE as `.pdf`) | **Yes** |
| `encrypt` | Download → AES-256-CTR with the lab key → upload as `<name><suffix>` → delete the original | **Yes** |
| `drop_note` | Write a harmless text "ransom note" into each touched folder | **Yes** |
| `sleep` | Advance the simulated clock without doing anything | **Yes** |
| `publish_feed` | Not an SMB op: write a fabricated MalwareBazaar-format CSV into `feeds/inbox/` (§5.7) | **Yes** |
| `raise_incident` | Not an SMB op: write an incident record into `incidents/inbox/` (§5.9) | **Yes** |
| `rename`, `delete`, `read`, `mkdir`, `archive`, `copy_in` | See the scenario catalog | Later (BL-05) |

Common parameters for all file actions: `share`, `paths` (glob or list), `count`, `rate` (ops/second), `jitter`.

#### 5.3.3 Scenarios
**v1: `dwell-window` only.** This is the plan's demo story:

| Sim time | Actor (as) | Action | Why |
|---|---|---|---|
| D1 08:00 | client-admin (admin) | `seed` corpus pack `default-42` | Baseline data on both shares |
| D1 09:00–17:00 | client-jdoe, client-asmith | `edit` about 6 finance docs, office style | Normal activity (and tests temp-file-then-rename) |
| **D1 09:14** | **client-attack (svc-backup)** | **`plant` benign PE as `finance/Q3/Q3-Report.pdf`** | The dwell-window payload |
| D2–D6, business hours | client-jdoe, client-asmith | `edit` about 4 docs/day | Noise inside the dwell window |
| **D5 10:00** | — | **`publish_feed`**: SHA-256 of `benign_pe` + 200 decoy hashes | Threat intelligence catches up (B7 → B8) |
| **D7 02:00** | **client-attack (svc-backup)** | **`encrypt` `finance/**` (seeded files only, rate 20/s), then `drop_note`** | The attack |
| **D7 02:05** | — | **`raise_incident`**: detection = this step's time; user, IP, machine from the encrypt step; admin restore time `D1 00:00` | Stand-in for Ransomware Protection + admin input (B9 → B10) |

`scenarios/dwell-window.toml` (abridged):
```toml
name        = "dwell-window"
description = "Plant a disguised executable on Day 1, encrypt the finance share on Day 7"
sim_epoch   = "2026-11-02T00:00:00Z"
corpus      = "default-42"

[[step]]
at = "D1 08:00"
actor = "client-admin"
action = "seed"

[[step]]
at = "D1 09:14"
actor = "client-attack"
as = "svc-backup"
action = "plant"
share = "finance"
path = "/Q3/Q3-Report.pdf"
artifact = "benign_pe"

[[step]]
at = "D5 10:00"
action = "publish_feed"
include_artifacts = ["benign_pe"]   # their SHA-256 go into the feed
decoys = 200                        # deterministic random hashes, never match

[[step]]
id = "attack"
at = "D7 02:00"
actor = "client-attack"
as = "svc-backup"
action = "encrypt"
share = "finance"
paths = "**"
rate = 20

[[step]]
at = "D7 02:05"
action = "raise_incident"
from_step = "attack"                # user, IP, machine, volume copied from that step (what the product detects)
restore_time = "D1 00:00"           # admin: latest moment with no attacker presence (required, D-22)
```

**Catalog (not built in v1; added on a need basis, BL-05):** `smash-and-grab`, `slow-drip`, `extension-only`, `wiper`, `staging`, `compromised-account` (U8; **first in line for v2**, D-12), `disguised-files`, `normal-week`.

#### 5.3.4 The simulated clock: what it is and why it exists
**Problem.** The demo story spans a week (Day 1 plant, Day 5 feed match, Day 7 encryption), and the dwell-window logic later asks questions like "what did this user write in the 30 days before the incident?". We can't wait a week on stage, and the real audit-log timestamps will all be *today, a few minutes apart*.

**Solution.** Every event carries two timestamps:
- `ts`: real wall-clock time, from the audit log (what actually happened).
- `sim_ts`: the **story time**, used by everything downstream (ledger, dwell-window query, timeline UI).

**How it works.**
1. A scenario starts at `sim_epoch` (e.g. `2026-11-02T09:00Z` = "D1 00:00"). Each step's `at` (`D1 09:14`, `D7 02:00`) is an offset from that epoch.
2. Just before executing a step, `dwta-sim` writes `run/simclock.json`:
   `{"run_id": "...", "sim_now": "2026-11-02T09:14:00Z", "wall_anchor": "2026-11-03T06:12:44Z"}`
   This means: "the real moment 06:12:44 corresponds to story time 09:14:00".
3. While the step runs (it may take a few seconds), the story clock ticks with real time: `sim_ts = sim_now + (event_wall_ts − wall_anchor)`.
4. The collector reads `simclock.json` when it stamps each event. It picks the clock entry whose `wall_anchor` is the latest one ≤ the event's `ts`, because the file is append-history (see SIM-04). That way a lagging collector still stamps correctly.
5. The next step (e.g. `D7 02:00`) writes a new entry, which jumps story time six days forward in one real second.

Without a scenario running (e.g. a file copied via Finder), `sim_ts = ts`.

#### 5.3.5 Benign corpus: built once, outside the simulation loop
**Decision.** Corpus generation is a **separate, one-time build step**, not part of a simulation run. Its output is a **corpus pack**: a folder of files plus a `manifest.json` (share-relative path, type, size, SHA-256). Scenarios refer to a pack by id (`corpus = "default-42"`). The `seed` action only *uploads* a pack.

```
dwta-sim corpus build --profile default --seed 42   # -> lab/corpus/default-42/{files…, manifest.json}
dwta-sim corpus verify default-42                    # re-hash every file against the manifest
```

**Why this option over the alternatives:**
| Option | Problem |
|---|---|
| Generate fresh files inside every run | Every run has different hashes, so ground truth and results can't be compared across runs, and runs get slower |
| Generate once, by hand, no generator | Can't be rebuilt on another Mac or after cleanup |
| **Deterministic generator, run once, pack reused** (chosen) | Same `--seed` gives byte-identical files (AT-15), so a pack can be rebuilt anywhere. Runs are fast, and the pack doubles as the **seed store** that `restore`/`reset` re-upload from (SIM-08) |

**Do we need several generators?** Not for v1. The collector and dwell-window story don't depend on what the files contain, only that they are realistic (correct magic bytes). Different packs become useful **later**, to evaluate the classifier and false positives: a document-heavy pack, a software-share pack, a 1,000-file pack for the overhead test (AT-12), or a pack made from a real but sanitized file sample. Because a pack is just "folder + manifest", any of those sources plugs in without simulator changes.

Contents of `default`: about 40 files across `finance/{Q1..Q4}/` (minimal valid PDF, DOCX, XLSX, CSV, TXT, PNG) and about 10 in `software/tools/` (benign PE/ELF stubs, ZIP), each with correct magic bytes. Files that `edit` produces during a run are *not* part of the pack; they are derived deterministically from the run seed and recorded in ground truth.

#### 5.3.6 Safe artifacts (require InfoSec approval)
| Artifact | Used as | Source | Required in v1 |
|---|---|---|---|
| Benign PE stub (valid `MZ`/`PE\0\0` headers; the entry point just returns) | `Q3-Report.pdf` (disguised) | Generated by script; checked into `lab/artifacts/` | **MUST** |
| EICAR test string | `notes.txt` (AV baseline, later) | Generated from the published string | SHOULD |
| Harmless macro document (`.docm`, see note) | `Budget.docm` | Produced once by Security eng. in Word | Later (D-11); not needed by `dwell-window` |
| Shortcut file (`.lnk` pointing to `notepad.exe`) | `Payroll.lnk` | Generated by script | Later |

**What the harmless macro document is.** A `.docm` is a Word document that is allowed to contain **macros**: small VBA programs stored inside the file (`word/vbaProject.bin`). Malicious macros are a classic way attackers plant code on shares: the file looks like a normal document, and the code runs when someone opens it and enables macros. Our test version contains one macro that only shows a message box (`MsgBox "DwTA test"`). It has no file, network or shell access. It exists so the later classifier (oletools) can prove it detects *"document with macro code"* without us handling real malware. It needs Word to create, which is why someone must produce it once and InfoSec must approve it.

#### 5.3.7 CLI
```
dwta-sim --instance dwta1 list                          # list scenarios and actions
dwta-sim --instance dwta1 validate scenarios/x.toml     # schema + safety checks, no execution
dwta-sim --instance dwta1 run dwell-window [--seed 42] [--speed max|realtime]
dwta-sim --instance dwta1 step plant --actor client-attack --as svc-backup --share finance \
         --path /Q3/Q3-Report.pdf --artifact benign_pe --at "D1 09:14"   # ad-hoc single action
dwta-sim --instance dwta1 restore <run_id>              # decrypt / undo everything the run did
dwta-sim --instance dwta1 reset                         # shares back to the seeded state
```

#### 5.3.8 Requirements
| ID | Requirement |
|---|---|
| SIM-01 | MUST be a host-side, **stdlib-only** Python 3.14-compatible CLI (§3.1), driving clients only through `docker compose -p <INSTANCE> exec`. |
| SIM-02 | MUST execute every file operation from the step's `actor` container, using the credentials named by `as` (default: the actor's own identity) and the actor's NetBIOS name. |
| SIM-03 | MUST load scenarios from **TOML** (D-01; stdlib `tomllib`), validate them (unknown action, unknown actor or identity, time going backwards, paths outside allowed shares) **before** any SMB operation, and fail with a clear message. |
| SIM-04 | MUST append a clock entry to `run/<run_id>/simclock.json` (and update `run/simclock.json` → current run) before each step, per §5.3.4. It MUST never rewrite earlier entries. |
| SIM-05 | MUST append one ground-truth record per file operation to `run/<run_id>/groundtruth.jsonl` (§6.2), *after* the SMB operation succeeds. Failed operations are recorded with `"result":"error"`. |
| SIM-06 | Actions MUST be pluggable: one module per action implementing `plan(step) -> [ops]` and `execute(op)`. A new action MUST NOT require changes to the runner. |
| SIM-07 | **Safety:** MUST refuse to run unless the target is the lab's `samba` container on the lab network. It MUST only use payloads from the generated corpus or `lab/artifacts/` (approved list). Destructive actions (`encrypt`, `delete`, `rename`) MUST be limited to files the run itself seeded or that exist in the seed manifest. There is no `--force`. |
| SIM-08 | **Reversibility:** `restore <run_id>` MUST undo the run (decrypt, rename back, re-upload deleted files from the corpus pack) with byte-identical SHA-256s. `reset` MUST return the shares to the seeded state. |
| SIM-09 | MUST be deterministic for a given `--seed` (file names, sizes, op order, jitter). |
| SIM-10 | `--speed max` (default) runs steps back to back. `--speed realtime` waits real time between steps (soak tests; not for the demo). |
| SIM-11 | The `dwell-window` scenario SHOULD complete in under 3 minutes with `--speed max`. |
| SIM-12 | MUST ship the `dwell-window` scenario (§5.3.3). Catalog scenarios are out of scope for v1. |
| SIM-13 | MUST build corpus packs with a separate `corpus build` command (§5.3.5). The same profile and seed MUST produce byte-identical files and manifest. Runs MUST NOT generate corpus files. |

**Acceptance:** AT-07, AT-10, AT-11, AT-15.

---

### 5.4 B4 — Collector

**Purpose.** Turn raw Samba audit lines into a clean, attributed, de-duplicated stream of **file events**, without ever slowing a write.

#### 5.4.1 Runtime
| ID | Requirement |
|---|---|
| COL-01 | MUST run as container `collector` (`python:3.12-slim`, stdlib + `watchdog` only) on the lab network, mounting `logs/` read-only, `data/` read-only and `run/` read-write. |
| COL-02 | MUST be fully asynchronous to Samba: it only reads the log file, and a stopped or slow collector MUST NOT affect SMB writes (verified by AT-12). |

#### 5.4.2 Parsing and correlation
| ID | Requirement |
|---|---|
| COL-03 | MUST parse every audit line into `{ts, user, client_ip, client_name, share, op, result, args}`. Unparseable lines go to `run/collector-deadletter.log` and increment a counter, never crash the collector. |
| COL-04 | MUST track open file handles per `(smbd pid, path)` and emit **one** `write_close` event when a handle that was created or written is closed. Opens and closes without a write emit nothing. |
| COL-05 | MUST emit `rename` (with `prev_path`), `delete` and `mkdir` events. |
| COL-06 | **Temp-file-then-rename:** writes to paths matching the temp patterns (`~WRL*.tmp`, `~$*`, `*.tmp`, `.~lock*`) MUST be held for up to `RENAME_WINDOW` (default 5 s). If a rename to a non-temp path follows, emit one `write_close` on the **final path** (with `via_rename=true`) and suppress the temp event. If not, emit the temp-path event when the window expires. |
| COL-07 | MUST persist its read position (`inode`, `offset`) in `run/collector.state` and resume after restart without loss or duplicates, including across log rotation. |
| COL-08 | `event_id` MUST be deterministic (SHA-256 of `source\|ts\|op\|share\|path\|user\|pid`), so any replay is idempotent downstream. |
| COL-09 | MUST stamp `sim_ts` using the simulated-clock rules in §5.3.4 (latest clock entry with `wall_anchor` ≤ event `ts`) and keep the original wall-clock `ts`. If there is no clock, `sim_ts = ts`. |
| COL-10 | Latency: p95 from SMB close to event emission ≤ 2 s (≤ `RENAME_WINDOW` + 2 s for held temp writes). |

#### 5.4.3 Fallback event source (Python file watcher)
| ID | Requirement |
|---|---|
| COL-20 | MUST implement a `watcher` source (`watchdog` / inotify on the `data/` mount) behind the same internal `EventSource` interface as the audit tailer. |
| COL-21 | Source selection by `COLLECTOR_SOURCE = audit \| watcher \| both`. In `both` mode, watcher events that match an audit event (same share and path, within 3 s) are dropped. Unmatched ones are emitted with `source="watcher"`, `user=null`, `client_ip=null`. |
| COL-22 | Default is **`both`** (D-03): audit gives attribution, the watcher cross-checks completeness. AT-12 still measures the overhead. |

**Why a watchdog fallback is needed.** The audit log is the *only* source of who and from where, but it is also the riskiest part of the build:
1. **Configuration risk.** Where `full_audit` writes (syslog vs smbd log), the op names and the line format all differ between Samba versions (V-01). If we can't get clean lines by Gate 1, the pipeline has no input.
2. **Correlation risk.** Real clients write temp files and rename them (Office, Explorer, some Finder copies). The audit stream shows many opens, writes, renames and closes per user save, and bugs here cause missed or duplicate events (the plan's top risk).
3. **Completeness check.** The watcher sees the *final state on disk* regardless of the SMB client. In `both` mode it acts as a cross-check that the tailer missed nothing (AT-07 recall), which is valuable even when the tailer works.
4. **Cost of the fallback.** It is small: about 50 lines behind the same interface. It loses attribution (no user, no IP), so it keeps the demo *running* but weakens the story. That trade-off is exactly why it is a fallback and not the primary source.

Decision (D-03): run **both**. The tailer is primary; the watcher is the cross-check, and the fallback if audit parsing fails.

#### 5.4.4 Output
| ID | Requirement |
|---|---|
| COL-30 | MUST emit events conforming to `dwta.fs_event/v1` (§6.1) through an internal `EventSink` interface with one method, `emit(event)`. |
| COL-31 | v1 MUST implement `JournalSink` (§5.4.6) and `StdoutSink` (debugging), per D-09. |
| COL-32 | `client_origin` MUST be `"lab"` when `client_ip` is inside `LAB_SUBNET` and not the gateway, otherwise `"host"`. |

#### 5.4.5 Ignore rules
| ID | Requirement |
|---|---|
| COL-40 | MUST ignore `.DS_Store`, `._*`, `Thumbs.db`, `desktop.ini`, and the `:AFP_*`/`:com.apple.*` stream names produced by `vfs_fruit`. The list is configurable. |
| COL-41 | MUST count ignored events in a metric, never drop them silently. |

#### 5.4.6 Storage and volume (D-09)
**The collector is a stream processor, not a database.** It reduces a high-volume, noisy audit stream into a much smaller event stream and hands it on. Querying ("what did svc-backup write during the dwell window before the incident?") is the job of the **ledger** (B6, §5.6; the plan's *Artifact ledger*: a SQLite table with one row per file written, holding its type, hash, writer and IP).

**Volume.** One user save produces about 8–15 audit lines (open, write × n, close, renames, stat noise), but only **one** event. Read ops are not audited (D-04).
| Situation | Audit lines | Events | Raw size |
|---|---|---|---|
| Dwell-window demo run | ≈ 2 k | ≈ 150 | < 1 MB |
| 1,000-file copy (AT-12) | ≈ 12 k | 1,000 | ≈ 3 MB |
| Busy production-like share: 1 M writes/day | ≈ 10 M/day | ≈ 1 M/day | ≈ 2.5 GB/day audit → ≈ 400 MB/day events (≈ 60 MB gzipped) |

**Three tiers, each with one owner:**
| Tier | What | Format | Owner | Kept | Queried how |
|---|---|---|---|---|---|
| 1. Raw audit log | Every Samba op | `logs/audit.log`, rotated at 50 MB | Samba (B2) | A few rotations | Not queried; debugging only |
| 2. **Event journal** | Every `fs_event` | Append-only **JSONL segments** `run/journal/YYYY-MM-DD/HH-<n>.jsonl`, rolled hourly or at 64 MB, gzipped when closed | **Collector (B4)** | Configurable (POC: forever) | Downstream blocks read it like a log: segment + offset. Ad hoc: **DuckDB** queries JSONL/gzip directly with SQL, no load step |
| 3. Ledger | One row per file version that matters | **Postgres 16**, indexed on `(volume, user, sim_ts)`, `(sha256)`, `(path)` | Ledger (B6) | At least the dwell-window horizon | SQL; the incident engine's dwell-window query |

**Why not write straight into a relational DB from the collector?**
- The journal is a durable buffer. If the classifier or ledger is down or being rewritten, nothing is lost, and the `event_id`s make replay idempotent (COL-08). This keeps Samba-side capture independent of everything downstream (the "never in the write path" principle).
- Appending to a file is the cheapest possible write, and it handles bursts (an encryption run is thousands of events per minute).
- The ledger only needs the *interesting* subset, enriched by the classifier (type, hash). Storing every raw event in the ledger would mix two jobs.
- Postgres handles the ledger comfortably at POC scale: tens of millions of rows with those indexes answer the dwell-window query in milliseconds. Production scale (many Edges) might swap the journal for Kafka or similar. Both are out of scope here; the schema is what carries over.

| ID | Requirement |
|---|---|
| COL-50 | `JournalSink` MUST append each event as one JSON line and `fsync` at least every 1 s or every 500 events. A segment is rolled hourly or at 64 MB, then gzipped. |
| COL-51 | MUST write `run/journal/INDEX` (segment list with first/last `event_id` and `sim_ts`), so consumers can find a starting point without scanning. |
| COL-52 | Replaying the same audit log MUST produce byte-identical journal content (deterministic `event_id`, stable field order). |

**Acceptance:** AT-05, AT-07 to AT-09, AT-12, AT-13, AT-16.

### 5.5 B5 — Classifier interface

**Purpose.** Judge each written file **by content**, at write time, asynchronously: what it really is, whether its name lies, and whether it carries macros or rule hits. Hash it so later threat intelligence can find it again without a rescan. This is "Day 1" of the story.

#### 5.5.1 How it works
1. Consumes the event journal (`run/journal/`) from its last committed offset. It handles `write_close`, `rename` and `delete`.
2. For `write_close`, it opens the file from the **read-only `data/` mount** (D-15), never over SMB. Reasons: no extra SMB session, no extra audit noise for the collector to filter, faster, and closest to Edge, which reads its local cache.
3. Runs **one content pass** (§5.11): a single streamed read that computes **SHA-256** and feeds the same bytes to the **typers** and **analyzers**. Files over 64 MB get head + tail only and no hash.
4. Applies the **extension-mismatch** and **write-time risk** rules, writes one `artifacts` row plus any alerts, and commits the journal offset **in the same database transaction** (exactly-once).
5. **Version linking (FW-L1, §5.11.6):** for each `write_close` the classifier looks up the logical file at `(volume, path)` in `files`. If it exists and is present, the new artifact becomes its next version (`version_no + 1`, `change_kind = overwrite`, `prev_artifact_id`, `prev_sha256`, `prev_type_*`). Otherwise a new `files` row is created (`change_kind = new_file`, version 1). If the hash equals the previous version's, the row is marked `same_content = true` and no write-time alert is repeated.
6. `rename` moves `files.current_path` (the `file_id` stays the same; a rename onto an existing path ends that file's life as `replaced`). `delete` sets `files.status = deleted`. Neither reads content.

#### 5.5.2 Plug-in interfaces
```python
class Typer(Protocol):            # "what is this file?"
    name: str                     # "rules" | "magika"
    version: str
    def type(self, f: FileSample) -> TypeVerdict: ...   # type_class, label, confidence (0..1), evidence

class Analyzer(Protocol):         # "what is inside it?"
    name: str                     # "oletools" | "yara-x"
    def applies_to(self, type_class: str) -> bool: ...
    def analyze(self, f: FileSample) -> list[Signal]: ...   # Signal(id, severity, detail)
```
`FileSample` gives `volume`, `path`, `ext`, `size`, `head` (first 64 KiB), `tail` (last 64 KiB), and `open()` for streaming. New typers or analyzers (e.g. TLSH, entropy, EMBER in v2) plug in without pipeline changes.

#### 5.5.3 Implementations in v1
| Kind | Name | What it does | Layer |
|---|---|---|---|
| Typer | `rules` | Magic-byte signature table + structural checks (`specs/magic.toml`): `MZ`+`PE\0\0`, ELF, Mach-O, `%PDF`, OLE2, ZIP→OOXML (`[Content_Types].xml`, `vbaProject.bin`), LNK header, PNG/JPEG, shebang scripts, EICAR string | **Rules (must)** |
| Typer | `magika` | Google Magika deep-learning file-type model (local, bundled, CPU). Label + score mapped to the same taxonomy via `specs/magika_map.toml` | **AI** |
| Analyzer | `oletools` | `olevba` on OLE/OOXML documents: has macros, auto-exec triggers, suspicious keywords | Rules (must) |
| Analyzer | `yara-x` | YARA-X rules from `specs/yara/` (LNK pointing to an executable, script droppers, PE in a document container, EICAR) | Rules (must) |

**Both typers always run** (when Magika is available), and both results are stored. The "rules only" vs "rules + AI" toggle is therefore a *view* over the same ledger, not a re-run:
- **rules** mode uses `type_rules`.
- **rules_ai** mode uses `type_ai` when Magika's score is ≥ `MAGIKA_MIN_SCORE` (default 0.8), and falls back to `type_rules` otherwise.

#### 5.5.4 Type taxonomy (`type_class`)
`pe_binary` · `elf_binary` · `macho_binary` · `script` · `lnk_shortcut` · `office_macro` · `office_doc` · `pdf` · `archive` · `image` · `text` · `test_signature` (EICAR) · `unknown`.
**Executable classes** = `pe_binary`, `elf_binary`, `macho_binary`, `script`, `lnk_shortcut`.

#### 5.5.5 Rules
| ID | Rule | Result |
|---|---|---|
| R-EXT | Extension → expected classes from `specs/ext_map.toml` (e.g. `.pdf → {pdf}`, `.docx → {office_doc}`, `.docm → {office_macro, office_doc}`, `.exe → {pe_binary}`). `ext_mismatch = true` if the detected class is not expected; `null` for unknown extensions | Field on artifact |
| R-W1 | Executable class on a **document** share | Alert `write_time`, **medium** |
| R-W2 | R-W1 **and** `ext_mismatch` (a disguised executable) | Alert `write_time`, **high**; replaces R-W1. Critical is reserved for feed matches and incidents |
| R-W3 | `office_macro` with auto-exec trigger on a document share | Alert `write_time`, **medium** |
| R-W4 | YARA-X hit | Alert `write_time`; severity from rule metadata |
| R-W5 | SHA-256 already in `feed_entries` (known-bad at write time) | Alert `feed_match`, **critical** (ownership: B8, §5.8; severity in `rules.toml`, like the retroactive `FEED-MATCH`) |
| R-W6 | **Type changed between versions:** a file's previous version was a non-executable class and the new version is an executable class (e.g. a real `Budget.pdf` overwritten with a PE). Uses `prev_type_*` from the ledger, with no extra read | Alert `write_time`, **high**; cites both versions |

Share kind (`document` / `software`) comes from share config (§5.2.3: `finance` = document, `software` = software). An executable on the software share raises no R-W1, by design. That is the false-positive control.

#### 5.5.6 Requirements
| ID | Requirement |
|---|---|
| CLS-01 | MUST run as container `classifier` (`python:3.12-slim` + `oletools`, `yara-x`, `magika`; wheels baked into the image at prep time so the build runs offline), with `data/` mounted **read-only** and a ledger connection (role `dwta_classifier`). |
| CLS-02 | MUST never block or slow SMB writes: it only consumes the journal (asynchronous, after file close). |
| CLS-03 | MUST obtain all content facts through the `ContentPass` interface (§5.11): **one full read per version** for files ≤ `hash_size_cap_bytes` (64 MB), producing SHA-256, typing input and analyzer results together. Above the cap: head + tail windows only, `sha256 = null`, `analysis = "partial"`. |
| CLS-04 | If the file is gone when read: write the artifact row with `status = gone_before_read` and `sha256 = null`. If its size differs from the event, the file was overwritten before it was read: set `changed_since_event`, keep `sha256` and type **empty** and raise **no** write-time alerts for this version, because those bytes belong to the later write, whose own event classifies them. |
| CLS-05 | Ledger write and journal offset commit MUST happen in one transaction. A restart MUST neither lose nor duplicate artifacts (unique `event_id`). |
| CLS-06 | If Magika fails to load or crashes, rules mode MUST continue unaffected and `type_ai = null` (the AI never takes down the core). |
| CLS-07 | Every alert and artifact MUST record which typer and rules produced it (`classifier`, `reasons`) so the incident engine and a later narrator can cite them. |
| CLS-08 | Latency: p95 from journal append to ledger commit ≤ 1 s for files ≤ 10 MB on the reference Mac. |
| CLS-09 | Taxonomy, ext map, magic table, Magika map, YARA rules **and rule severities (R-W1…R-W6)** live in `specs/` (TOML/YARA files), **not in code**. They are versioned, and `classifier_version` (code version + config hash) is stored per artifact. Changing a severity or an extension mapping needs a config edit and a service restart, never a code change. |

**Acceptance:** AT-17 to AT-28.

---

### 5.6 B6 — Artifact ledger

**Purpose.** The system's memory: every file version written, what it is, who wrote it, from where and when, plus everything later learned about it. It is the **only** state the matcher and incident engine use, and with `fs_event` it forms the product **integration contract**.

| ID | Requirement |
|---|---|
| LED-01 | One **Postgres 16** container `ledger` (`postgres:16-alpine`, arm64), database `dwta`, data on the Docker named volume `<INSTANCE>_pgdata`. **No host port by default**, because 5432 is already used on the reference Mac. `LEDGER_HOST_PORT` (first free in 15432–15499) publishes it on `127.0.0.1` only, when asked. |
| LED-02 | Schema as in §6.3, managed by numbered plain-SQL migrations (`ledger/migrations/NNN_*.sql`), run by a one-shot `ledger-migrate` service before the others start. Every service checks `meta.schema_version` at start. |
| LED-03 | One database role per writer: `dwta_classifier` (artifacts, write-time alerts, its offsets), `dwta_matcher` (feed tables, match alerts), `dwta_incident` (incidents, plans, incident alerts), plus a read-only `dwta_reader` for the API/UI and analysis. Each role can write only its own tables (`GRANT`). Passwords come from `.env`. |
| LED-04 | Alerts are written to the `alerts` table **and** appended to `run/alerts.jsonl` (D-16). Each alert carries a deterministic `alert_id`, and inserts use `ON CONFLICT DO NOTHING`, so a replay never duplicates. |
| LED-05 | **Portable SQL:** stick to standard types and constraints so the schema could move to another engine. Postgres-specific features are limited to `JSONB` columns and `ON CONFLICT`. |
| LED-06 | `dwta-ledger export` runs `pg_dump` to `run/ledger-<ts>.sql` for snapshots. `psql` and DuckDB (`ATTACH … (TYPE postgres)`) can connect through `LEDGER_HOST_PORT` for inspection. |
| LED-07 | Dwell-window query (§5.10) MUST use the `(volume, sim_ts)` index and run in < 100 ms for 1 M artifacts. Container RAM ≤ 256 MB (`shared_buffers = 64MB`). |

---

### 5.7 B7 — Threat feed (fabricated)

**What a threat feed is here.** A list of SHA-256 hashes that threat intelligence has newly confirmed as malicious. In real life it comes from abuse.ch MalwareBazaar or a vendor, and it arrives **after** an attacker's file is already sitting on the share. In the lab we fabricate it, because our planted file is harmless and nobody would ever list it. The feed's job is to let the matcher (B8) say "a file we already have is now known-bad". The classifier never sees the feed.

| ID | Requirement |
|---|---|
| FEED-01 | Format: **MalwareBazaar CSV export** column set: `first_seen_utc, sha256_hash, md5_hash, sha1_hash, reporter, file_name, file_type_guess, mime_type, signature, clamav, vtpercent, imphash, ssdeep, tlsh`, with `#` comment header lines (exact columns confirmed against a current export: V-07). |
| FEED-02 | `publish_feed` (simulator action) MUST write `feeds/inbox/feed-<sim date>-<n>.csv` **atomically** (temp file + rename). Hashes come from `include_artifacts` (computed from the approved artifact files) plus `decoys` deterministic random hashes. `first_seen_utc` = the step's simulated time. `reporter = "dwta-lab"`, `signature = "DwTA.TestFamily"`. |
| FEED-03 | MUST also record the published feed in ground truth (`action = publish_feed`, list of hashes, which ones are expected to match). |
| FEED-04 | v1 publishes feeds **only** through the simulator (D-14). A standalone mock feed service (BL-09) and real MalwareBazaar import (BL-11) come later behind the same inbox contract. |

---

### 5.8 B8 — Feed matcher

**Purpose.** Re-check the past cheaply: when new threat intelligence arrives, find matching files by **hash lookup in the ledger**, never by rescanning the share. This is "Day 5".

| ID | Requirement |
|---|---|
| MAT-01 | MUST run as container `matcher` with **no share data mounted** (it is physically unable to rescan). It mounts `feeds/` and connects to the ledger (role `dwta_matcher`). |
| MAT-02 | MUST poll `feeds/inbox/` (1 s), parse and validate each CSV, skip `#` lines, normalize hashes to lowercase hex, and reject malformed rows to `feeds/rejected/` with a reason. |
| MAT-03 | MUST upsert into `feed_entries` (unique `(feed_id, sha256)`), then match all new hashes with **one indexed join** against `artifacts.sha256`, and write `feed_matches` (unique `(artifact_id, sha256)`) plus a **critical** `feed_match` alert per matched artifact. `sim_ts` of the alert = the feed's `first_seen_utc`. |
| MAT-04 | **Reverse path:** after each classifier commit, artifacts whose hash is already in `feed_entries` MUST also be matched (R-W5). The matcher does this by polling for unmatched artifacts newer than its high-water mark, so the classifier never calls it directly. |
| MAT-05 | Idempotent: re-dropping the same feed file MUST create no new entries, matches or alerts. Processed files move to `feeds/processed/`. |
| MAT-06 | Matches on deleted artifacts are still recorded (with `artifact_status`), because the incident engine needs them for history. |
| MAT-07 | Performance: a 100 k-hash feed is ingested and matched in < 5 s on the reference Mac. |

**Acceptance:** AT-29 to AT-31.

---

### 5.9 B9 — Simulated incident

**What it stands for.** DwTA does **not** detect ransomware. In the product, Ransomware Protection already raises an incident. DwTA *consumes* that incident and answers "what did the attacker leave behind?". In the lab, B9 stands in for the product's detector (D-18), plus the one thing the admin has to tell us: the restore time.

**Two inputs, two owners** (D-24):
| Input | From | Fields |
|---|---|---|
| Incident | The product's detector (assumed, D-24) | `detected_sim_ts`, `user` (who initiated it), `client_ip`, `client_name` (machine), `volumes[]` |
| Restore time | The admin | `restore_time_sim_ts`: the **latest moment the admin believes had no attacker presence**. It is the snapshot Targeted Restore uses, and it is also our estimate of when the attacker arrived. The real start is never known |

| ID | Requirement |
|---|---|
| INC-01 | Incident record `dwta.incident/v1` (§6.5): `incident_id`, `source` (`scenario`/`cli`), `kind` (`ransomware`), `detected_sim_ts`, `user`, `client_ip`, `client_name`, `volumes[]`, `restore_time_sim_ts` (may arrive later), `description`. There is **no start-time field**: the product doesn't provide one. |
| INC-02 | **Scenario:** `raise_incident` with `from_step` copies user, IP, machine and volume from the referenced step; `detected_sim_ts` = the raise step's own time; `restore_time` is a step parameter (the admin's answer), so the scenario delivers both inputs at once. |
| INC-03 | **CLI, two-stage like the product:** `dwta-incident raise --detected "D7 02:05" --user svc-backup --ip 10.250.1.66 --machine SRV-BKP01 --volume finance` creates the incident in state `awaiting_restore_time`. Then `dwta-incident set-restore-time <incident_id> "D1 00:00"` supplies the admin input and triggers the review. `raise … --restore-time …` does both at once. |
| INC-04 | All paths MUST write atomically to `incidents/inbox/<incident_id>.json` (updates replace the file); the incident engine is the only consumer. |
| INC-05 | The engine MUST stay robust if `user`, `client_ip` or `client_name` is missing (it ranks without that signal), even though D-24 assumes the product provides them. |
| INC-06 | No review runs until `restore_time_sim_ts` is set. A restore time ≥ `detected_sim_ts`, or older than `max_window_days`, MUST be rejected to `incidents/rejected/` with a reason. There is **no default** (D-22). |

---

### 5.10 B10 — Incident engine

**Purpose, in one sentence.** Given "an attack was detected at D7 02:05, by svc-backup from SRV-BKP01" and the admin's "restore to D1 00:00", list **before the restore** only the files written in between that look suspicious, ranked by suspicion score and level with their reasons, so the admin can mark each one **remove** (keep it out of the recovery) or **keep**. This is the core idea's steps ③–⑤.

#### 5.10.1 The window (D-22)
```
  D1 00:00  ◀ admin's RESTORE TIME (latest moment with no attacker presence) ─┐
  D1 09:14    attacker plants Q3-Report.pdf                                    │
  D2–D6       normal user edits                                                │  ONE WINDOW
  D7 02:00    encryption begins (true start, never known to us)               │  [restore time → detection]
  D7 02:01    (maybe) a 2nd payload                                            │
  D7 02:05  ◀ product DETECTS (time, user, IP, machine) ───────────────────────┘
```
- Window = `[restore_time_sim_ts, detected_sim_ts]`. **One window**: it covers the dwell period *and* the attack itself, so a second payload dropped during the attack is caught automatically.
- **Why the restore time is the right start:** Targeted Restore rolls affected files back to that moment. Anything written *after* it and not restored is still on the share afterwards: planted payloads, a second payload, even the attacker's own encrypted copies and ransom notes. That is exactly what DwTA must surface.
- **Caveat:** if the admin's restore time is too late (after the attacker's first plant), the plant falls outside the window. Proposed safety net: OQ-02.
- Candidates = artifacts on the incident's volumes with `sim_ts` in the window and status `present`, `renamed` or `changed_since_event`. Deleted ones are counted as "seen but gone".
- **Barrier:** before ranking, wait until the classifier has committed every journal event up to `detected_sim_ts` (timeout 30 s, then rank and flag `incomplete = true`).

#### 5.10.2 Attack outputs: one group, not hundreds of rows (D-25)
The encryption run creates many new files in the window (`*.dwtalocked` copies, ransom notes). They're suspicious but need **one** decision, not one per file. Before ranking, the engine moves them into a single **attack-outputs group**, using `specs/attack_outputs.toml`:
```toml
extensions   = [".dwtalocked", ".locked", ".encrypted", ".crypt"]
note_names   = ["README*.txt", "*DECRYPT*", "*RECOVER*", "*RANSOM*"]
replaced_original = { enabled = true, within_seconds = 60 }   # new "<X><suffix>" after "<X>" was deleted by the same user
```
The group shows as one review item with count, sample paths, writer and time range.

#### 5.10.3 Ranking rules (configured in `specs/ranking.toml`; proposed values in use until the team provides final ones, OQ-01)
```toml
# specs/ranking.toml: edited by security/PM, never hard-coded (ENG-07)
version   = "2026-11-03.2"
min_score = 30             # below this a file is not listed
[levels]                   # suspicion level shown to the admin (D-35)
high   = 100
medium = 60
low    = 30
max_window_days = 90       # sanity cap; the window comes from the admin's restore time (D-22)
tie_break = ["score:desc", "sim_ts:asc"]

[signals]
S-FEED     = { enabled = true, weight = 100 }
S-USER     = { enabled = true, weight = 40 }
S-IP       = { enabled = true, weight = 20 }
S-HOST     = { enabled = true, weight = 10 }
S-EXEC-DOC = { enabled = true, weight = 30 }
S-EXT      = { enabled = true, weight = 30 }
S-MACRO    = { enabled = true, weight = 15 }
S-YARA     = { enabled = true, weight = "from_rule" }   # YARA rule metadata, 10-30
S-TYPE-CHANGE = { enabled = true, weight = 30 }
S-AV          = { enabled = true, weight = 100 }   # step ⑤ rescan detected the file (v2)
```

| Signal | ID | Weight | Evidence cited |
|---|---|---|---|
| SHA-256 matched a threat feed | S-FEED | +100 | `feed_matches` row |
| Written by the incident's user | S-USER | +40 | `artifacts.user` = `incident.user` |
| Written from the incident's IP | S-IP | +20 | `artifacts.client_ip` = `incident.client_ip` |
| Written from the incident's machine | S-HOST | +10 | `artifacts.client_name` = `incident.client_name` |
| Executable class on a document share | S-EXEC-DOC | +30 | `type_class`, share kind |
| Extension mismatch | S-EXT | +30 | `ext`, `type_class`, `ext_mismatch` |
| Macro document with auto-exec | S-MACRO | +15 | oletools signal |
| YARA-X hit | S-YARA | rule weight (10–30) | rule id |
| File type changed into an executable since its previous version | S-TYPE-CHANGE | +30 | both artifact rows (`prev_artifact_id`) |
| Antivirus rescan of the flagged file detected it (step ⑤, §5.10.4) | S-AV | +100 | AV signature, engine, definitions version |

**Only files that need human input are listed** (D-23): files with `score ≥ min_score`, plus the attack-outputs group. Each item shows its **`suspicion_score`** (the sum of its signal weights) and **`suspicion_level`** from the `[levels]` bands: **high** ≥ 100, **medium** ≥ 60, **low** ≥ 30 (D-35). The attack-outputs group is always **high**. Normal edits by legitimate users score 0 and never appear. Items are ordered by the tie-break rule. Each stores its `reasons[]` (signal id, weight, evidence row ids), and a later narrator MUST cite only those.

#### 5.10.4 Step ⑤: antivirus rescan of the flagged files (D-36)
**Why.** A planted file usually passed antivirus on Day 1 because no signature existed yet. By the time of the incident, signatures may have caught up. Rescanning **only the flagged files** with today's signatures, just before the admin sees the list, adds independent evidence at a tiny read cost.

**Flow.** After scoring (§5.10.3) and before the list is written: every **file** item with `score ≥ min_score` is scanned once. The attack-outputs group is **not** scanned (encrypted copies tell the AV nothing). A detection adds **S-AV** to the item's score; levels and ranks are then recomputed. Every item shows its verdict.

```python
class AvScanner(Protocol):                       # B10 plug-in, like the classifier's typers
    name: str                                    # "none" (v1) | "clamav" (lab, v2) | "edge-av" (product)
    def scan(self, volume: str, path: str) -> AvVerdict: ...
# AvVerdict(verdict = "detected" | "clean" | "not_scanned" | "error", signature: str | None,
#           engine: str | None, definitions: str | None, scanned_at: datetime | None)
```

| ID | Requirement |
|---|---|
| AV-01 | The engine MUST call `AvScanner.scan` for every listed **file** item, before writing the review list and before the admin can see it. The attack-outputs group is skipped. |
| AV-02 | `detected` adds signal **S-AV** (weight in `ranking.toml`, default **+100**) with the signature name as evidence; the item is re-scored, re-levelled and re-ranked. `clean` adds nothing. |
| AV-03 | Every **file** item MUST carry `av_verdict`, `av_signature`, `av_engine`, `av_definitions` and `av_scanned_at`, shown to the admin next to the score. |
| AV-04 | The rescan MUST never block the review: per-file timeout (default 30 s), and if the scanner is unavailable the verdict is `not_scanned` (or `error`) and the review proceeds (like CLS-06 for Magika). |
| AV-05 | Read budget: this is the **only** content read after write time, limited to the listed files, each read **once per review** by the scanner (reason `av_rescan`, RB-R3 exception). A `replan` re-uses verdicts younger than `av_reuse_minutes` (default 60). |
| AV-06 | **v1:** the `none` scanner returns `not_scanned` for every item, so the contract, the review list and the UI field exist from day one. **v2:** `clamav` scanner = a ClamAV container (`clamd`, signatures pulled at prep time) with the share data mounted read-only, called by the incident engine over the lab network. **Product:** the Edge antivirus with its current definitions. |
| AV-07 | Lab demo (v2): a feed-only signature added to the ClamAV database for the benign test PE shows "clean on Day 1, **detected** on Day 7", which is the timing story in one line. |

#### 5.10.5 Admin decisions and the restore-exclusion manifest (D-33, D-34)
| Decision | Meaning | Executed later (API/UI) as |
|---|---|---|
| `pending` | Default; not yet reviewed | — |
| `remove` | **Keep this file out of the recovery** | **New file** (e.g. `Q3-Report.pdf`): moved to a locked quarantine folder, **never deleted**, reversible. **Overwritten file**: the attacker's version is moved to quarantine and Targeted Restore puts back the restore-time version (the version links show which). **Group**: applies to every member |
| `keep` | Admin judges it legitimate; it stays | Nothing; recorded with who and when, for tuning |

Each item shows `change_kind` (`new_file` / `overwrite`) and `version_no`, so the admin sees what `remove` will do.

**Flow in v1:**
1. `dwta-incident decide <incident_id> <item_id> remove|keep --by <admin>` records a decision (state `under_review`).
2. `dwta-incident finalize <incident_id> --by <admin>` is refused while any item is `pending`. Otherwise it locks the decisions (state `finalized`) and writes **`run/incidents/<id>/restore-exclusions.json`** (§6.7): the files to remove, with their reasons and who decided. That manifest is the hand-off to the restore step.
3. v1 **never touches files** (D-17). Executing the manifest, together with Targeted Restore, comes with API/UI (BL-06).

#### 5.10.6 Requirements
| ID | Requirement |
|---|---|
| ENG-01 | MUST compute the review list for **both** modes (`rules`, `rules_ai`) using the mode's `type_class` (§5.5.3), and store both, so the UI toggle needs no re-run. |
| ENG-02 | MUST write `incidents`, `review_items` and one `incident` alert (critical) to the ledger, and `run/incidents/<incident_id>/review-list.json`. |
| ENG-03 | In v1 the review is **advisory only**: no file is moved, locked, restored or deleted (D-17). Decisions are recorded and the restore-exclusion manifest is written. |
| ENG-09 | Incident states MUST follow `awaiting_restore_time → under_review → finalized` (or `rejected`). `finalize` MUST be refused while any item is `pending`. After `finalize`, decisions and the manifest are immutable; a changed restore time needs a new review (new `incident_id`). |
| ENG-11 | Step ⑤ AV rescan runs between scoring and writing the review list (§5.10.4, AV-01..07). |
| ENG-10 | Every review item MUST carry `suspicion_score` and `suspicion_level` from the `[levels]` bands in `ranking.toml` (config, not code). |
| ENG-04 | Deterministic: the same ledger state + incident gives a byte-identical review list. |
| ENG-05 | Re-running the same `incident_id` (e.g. a new restore time) replaces its list but **keeps decisions** already made on items that are still listed. A new `incident_id` gets its own list. |
| ENG-06 | Review generation p95 < 2 s for 1 M artifacts in the ledger. |
| ENG-07 | **All ranking settings are configuration, not code:** signal weights, enabled signals, `min_score`, tie-break order, `max_window_days` (`specs/ranking.toml`) and attack-output patterns (`specs/attack_outputs.toml`). Both files are re-read for **every** review, so a tuning change applies to the next run without a restart. |
| ENG-08 | Every review list MUST record the config it used (version + SHA-256 of both files). `dwta-incident replan <incident_id>` re-ranks with the current config for what-if tuning. |

**Acceptance:** AT-32 to AT-42.

### 5.11 Read budget and antivirus integration (cross-cutting; merged from `SPEC-read-budget-av-integration.md`)

**The point we advocate (D-26).** Nasuni Antivirus Protection (ClamAV) already reads every new or modified file **in full**. In the product, DwTA takes what it needs from **that** read: the SHA-256, the file type, YARA hits (ClamAV loads YARA rules natively) and macro detection (ClamAV parses OLE2/VBA). So **DwTA adds no file reads beyond what antivirus already does.** We assume ClamAV 1.5+ callbacks, or a small modification of the Edge's ClamAV integration, expose those results without extra reads (VERIFY-RB-1 to 3, OQ-03).

**In the POC (v1)** ClamAV isn't in the lab yet (v2, D-37), so the classifier's own `ContentPass` **stands in for the antivirus pass**: exactly one full read per file version, and everything else reuses its output.

#### 5.11.1 Content-pass interface
```python
class ContentPass(Protocol):
    name: str                                  # "dwta_single_pass" (v1) | "clamav_harvest" (product)
    def run(self, f: FileRef) -> ContentFacts: ...

@dataclass
class ContentFacts:
    sha256: str | None                         # None above the hash cap or if not provided
    size: int
    head: bytes                                # first 65,536 bytes
    tail: bytes                                # last 65,557 bytes (22-byte ZIP end record + 65,535-byte max comment)
    av_type: str | None                        # file type from the AV pass (product)
    signals: list[Signal]                      # oletools / YARA results from the same pass
    source: str                                # which pass produced these facts
```
| Implementation | Where | What it does |
|---|---|---|
| `dwta_single_pass` | **POC v1** | Streams the file once from the read-only mount. Feeds SHA-256, keeps head + tail, and runs oletools + YARA-X on the same bytes. Files ≤ 64 MB only; above that, head + tail reads only |
| `clamav_harvest` | **Product** (notes only, RB-P1–P5) | Consumes the antivirus pass's hash, type and rule results asynchronously. DwTA's own reads drop to zero (or tail-only for formats the AV doesn't identify, VERIFY-RB-3) |

Typers (rules, Magika) work on `head`/`tail` only, so they never need a read of their own.

#### 5.11.2 Which checks need what
| Check | Needs | Source in v1 |
|---|---|---|
| Magic-byte typing, Magika | Head (+ tail for ZIP/OOXML) | Pass `head`/`tail` |
| OOXML detection, macro **presence** in `.docm/.xlsm` | Tail (ZIP central directory) | Pass `tail` |
| Extension mismatch, R-W1, R-W2, R-W5 | Type + hash, no bytes | Ledger |
| YARA header rules (PE, LNK, EICAR) | Head | Pass `head` |
| **SHA-256** | Full file | Pass (one read) |
| **oletools deep macro analysis** (auto-exec, keywords; legacy `.doc/.xls`) | Full file | Same pass (no extra read) |
| **YARA content-string rules** | Full file | Same pass (no extra read) |

#### 5.11.3 Rules adopted in v1
| ID | Rule (RB source) | v1 |
|---|---|---|
| RB-R2/R4 | **One full read per version**, and only in the content pass. A version's hash is computed once and never recomputed; later steps use the ledger | MUST |
| RB-R1 | Above the 64 MB hash cap (FW-H2): read at most head 65,536 + tail 65,557 = **131,093 bytes** | MUST |
| RB-R3 | **Zero content reads** for derived work: feed ingest and matching, renames, deletes, ranking, incident review, narratives. **Single exception:** the step ⑤ AV rescan reads the *listed* files once per review (AV-05) | MUST (the matcher has no data mount, MAT-01) |
| RB-R5 | Only current files are read, never history (historical hunts run storage-side in the product) | MUST |
| RB-M1 | Evaluation mode **`immediate`** only (on close). `per_snapshot` is backlog (BL-13) | D-28 |

Hash policy (D-27): **every file ≤ 64 MB is hashed**, so a feed can match any file type. Known limit: files over 64 MB can never feed-match (they have no hash).

#### 5.11.4 Moved to v2 (with the ClamAV container, D-37)
Read accounting (`ReadMeter` reasons persisted in the ledger, RB-A1/A3), the ClamAV baseline measured with `rchar` and modeled (RB-A2), `/metrics/reads` (RB-A4) and the demo panel (RB-A5); BL-12. Expected product result: **DwTA's extra bytes read ≈ 0**, because everything comes from the AV pass. v1 keeps a minimal in-process read counter **for tests only** (AT-22 to AT-25).

#### 5.11.5 Requirements
| ID | Requirement |
|---|---|
| RBQ-01 | All file-content access in DwTA services MUST go through `ContentPass` (the classifier only). Any other content read fails review. |
| RBQ-02 | `dwta_single_pass` MUST stream the file in one sequential pass (no seeks back), feeding the hasher, the head/tail buffers and the analyzers. oletools gets the file from an in-memory buffer only when the type needs it (OLE2/OOXML ≤ 64 MB). |
| RBQ-03 | Settings in `specs/read_budget.toml` (config, not code): `head_bytes = 65536`, `tail_bytes = 65557`, `small_file_full_read_bytes = 131072`, `hash_size_cap_bytes = 67108864`, `mode = "immediate"`. |
| RBQ-04 | The test read counter MUST record bytes and reason (`pass`, `head`, `tail`) per version, so AT-22 to AT-25 can assert on it. |

#### 5.11.6 Follow-up writes (`FW-`): what happens when a known file is written again
| ID | Decision | v1 behavior |
|---|---|---|
| FW-K1 | Read window for files **over 64 MB**: head 65,536 + tail 65,557 bytes | Typing, Magika and header YARA from head/tail; no hash |
| FW-H1 | **Full SHA-256 for every version ≤ 64 MB** | One full read per version (the content pass) |
| FW-H2 | Hash cap **64 MB** (`hash_size_cap_bytes`) | Above it: `sha256 = null`, `analysis = partial` |
| FW-C3 | **No close coalescing** for now: every `write_close` is evaluated and read | Ten quick saves = ten versions = ten reads. Coalescing is backlog (BL-14) |
| FW-L1 | **Version tracking in the ledger**: a `files` table (one row per logical file) and version links on `artifacts` (`file_id`, `version_no`, `change_kind`, `prev_*`). **Metadata only; no content is ever stored.** Samba provides no versions; the classifier builds them from the event stream | Enables `change_kind` for the review list, the type-change rule (R-W6, S-TYPE-CHANGE) and same-content re-save detection |

| ID | Requirement |
|---|---|
| FW-01 | Version linking MUST happen in the same transaction as the artifact insert (CLS-05). The classifier processes events in journal order, so versions of a path never race. |
| FW-02 | A temp-file-then-rename save (collector `via_rename = true`) onto an existing path MUST become an `overwrite` version of that path's file, not a new file. |
| FW-03 | `same_content = true` versions MUST NOT raise new write-time alerts; they still count as versions (their `sim_ts` matters for the window). |
| FW-04 | R-W6 and S-TYPE-CHANGE MUST use only ledger data (`prev_type_*`), never a re-read of an older version (RB-R5). |

## 6. Data contracts (v1)

Contracts 6.1–6.7 are what the product would need to adopt. Everything else is replaceable.

### 6.1 `dwta.fs_event/v1`: collector output
This is the **raw, pre-classification** event. The plan's integration contract (`type_class`, `classifier`, `ext_mismatch`, `sha256`, `tlsh`) is added later by the classifier. The overlapping fields keep the same names (`ts`, `volume`, `path`, `user`, `client_ip`, `size`).

```json
{
  "schema": "dwta.fs_event/v1",
  "event_id": "9f3c…",                      // COL-08, deterministic
  "source": "audit",                        // audit | watcher
  "op": "write_close",                      // write_close | rename | delete | mkdir
  "ts": "2026-11-03T06:12:44.381Z",         // wall clock from the audit line
  "sim_ts": "2026-11-02T09:14:07Z",         // simulated clock (COL-09)
  "volume": "finance",                      // SMB share name
  "path": "/Q3/Q3-Report.pdf",              // share-relative, final path
  "prev_path": null,                        // set for rename and via_rename writes
  "via_rename": false,
  "user": "svc-backup",
  "client_ip": "10.250.1.66",
  "client_name": "SRV-BKP01",
  "client_origin": "lab",                   // lab | host
  "size": 245760,                           // stat() of the final file, null if gone
  "raw_ref": "audit.log@1048576"            // offset of the closing line, for traceability
}
```

### 6.2 `dwta.groundtruth/v1`: simulator answer key
**What it is.** `groundtruth.jsonl` is the simulator's **answer key**: one JSON line for every SMB operation it performed, saying exactly who did what, from where, at what simulated time, and which event the collector *should* produce. It is a test oracle only; production would never have it. It is used to:
- score the collector (AT-07: every ground-truth row must have exactly one matching event, so 100 % recall and no duplicates);
- later, score detectors (was every `malicious=true` row flagged, and how many `malicious=false` rows were flagged?).

```json
{"schema":"dwta.groundtruth/v1","run_id":"r-20261103-061240","scenario":"dwell-window","step":2,
 "op_id":"r-20261103-061240/0007","sim_ts":"2026-11-02T09:14:07Z","wall_ts":"2026-11-03T06:12:44Z",
 "actor":"client-attack","identity":"svc-backup","client_ip":"10.250.1.66","client_name":"SRV-BKP01",
 "action":"plant","volume":"finance","op":"write","path":"/Q3/Q3-Report.pdf","prev_path":null,
 "artifact":"benign_pe","sha256":"…","malicious":true,"result":"ok","expected_event":"write_close"}
```

### 6.3 Ledger schema (Postgres 16, portable SQL)
```sql
CREATE TABLE files (                    -- one "index card" per logical file; metadata only, never content (FW-L1)
  file_id            BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  volume TEXT NOT NULL, current_path TEXT NOT NULL,
  first_seen_sim_ts  TIMESTAMPTZ NOT NULL,
  latest_artifact_id BIGINT,             -- link to the newest version
  version_count      INTEGER NOT NULL,
  status TEXT NOT NULL);                 -- present | deleted | replaced
CREATE UNIQUE INDEX ux_files_live ON files(volume, current_path) WHERE status = 'present';

CREATE TABLE artifacts (                -- one row per file VERSION written (B5)
  artifact_id      BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  event_id         TEXT UNIQUE NOT NULL, -- from fs_event (idempotency)
  file_id BIGINT NOT NULL REFERENCES files, version_no INTEGER NOT NULL,
  change_kind TEXT NOT NULL,             -- new_file | overwrite
  prev_artifact_id BIGINT REFERENCES artifacts, prev_sha256 CHAR(64),
  prev_type_rules TEXT, prev_type_ai TEXT, same_content BOOLEAN NOT NULL DEFAULT false,
  volume TEXT NOT NULL, share_kind TEXT NOT NULL,          -- document | software
  path TEXT NOT NULL, current_path TEXT NOT NULL, ext TEXT,
  status TEXT NOT NULL,                  -- present|renamed|deleted|gone_before_read|changed_since_event
  size BIGINT, sha256 CHAR(64),
  type_rules TEXT, type_ai TEXT, ai_score REAL,
  ext_mismatch_rules BOOLEAN, ext_mismatch_ai BOOLEAN,
  signals JSONB,                         -- [{id, severity, detail, analyzer}]
  analysis TEXT NOT NULL DEFAULT 'full', -- full | partial
  "user" TEXT, client_ip INET, client_name TEXT, client_origin TEXT,
  ts TIMESTAMPTZ NOT NULL, sim_ts TIMESTAMPTZ NOT NULL, classified_at TIMESTAMPTZ NOT NULL,
  classifier_version TEXT NOT NULL);
CREATE INDEX ix_art_sha   ON artifacts(sha256);
CREATE INDEX ix_art_file  ON artifacts(file_id, version_no);
CREATE INDEX ix_art_win   ON artifacts(volume, sim_ts);
CREATE INDEX ix_art_user  ON artifacts(volume, "user", sim_ts);
CREATE INDEX ix_art_path  ON artifacts(volume, current_path);

CREATE TABLE feed_entries (feed_id TEXT, sha256 CHAR(64), first_seen_sim TIMESTAMPTZ, signature TEXT,
  file_name TEXT, reporter TEXT, imported_at TIMESTAMPTZ, PRIMARY KEY (feed_id, sha256));
CREATE INDEX ix_feed_sha ON feed_entries(sha256);

CREATE TABLE feed_matches (match_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  artifact_id BIGINT REFERENCES artifacts, sha256 CHAR(64), feed_id TEXT,
  matched_sim_ts TIMESTAMPTZ, matched_at TIMESTAMPTZ, UNIQUE (artifact_id, sha256));

CREATE TABLE incidents (incident_id TEXT PRIMARY KEY, source TEXT, kind TEXT,
  detected_sim_ts TIMESTAMPTZ NOT NULL,                     -- from the product
  "user" TEXT, client_ip INET, client_name TEXT, volumes TEXT[] NOT NULL,
  restore_time_sim_ts TIMESTAMPTZ,                          -- from the admin; window start
  state TEXT NOT NULL,                                      -- awaiting_restore_time | under_review | finalized | rejected
  finalized_by TEXT, finalized_at TIMESTAMPTZ,
  config_version TEXT, config_sha256 CHAR(64), incomplete BOOLEAN, created_at TIMESTAMPTZ,
  CHECK (restore_time_sim_ts IS NULL OR restore_time_sim_ts < detected_sim_ts));

CREATE TABLE review_items (item_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  incident_id TEXT REFERENCES incidents, mode TEXT, rank INTEGER,
  item_kind TEXT NOT NULL,                                  -- file | attack_outputs_group
  artifact_id BIGINT REFERENCES artifacts,                  -- null for the group
  group_members BIGINT[],                                   -- artifact ids, for the group
  change_kind TEXT,                                         -- new_file | overwrite (from artifacts)
  score INTEGER, suspicion_level TEXT,                      -- high | medium | low
  reasons JSONB,
  decision TEXT NOT NULL DEFAULT 'pending',                 -- pending | remove | keep
  decided_by TEXT, decided_at TIMESTAMPTZ,
  UNIQUE (incident_id, mode, item_kind, artifact_id));

CREATE TABLE alerts (alert_id CHAR(64) PRIMARY KEY,      -- deterministic hash
  kind TEXT, severity TEXT, mode TEXT,                    -- write_time|feed_match|incident ; rules|rules_ai|both
  artifact_id BIGINT, incident_id TEXT, rule_ids TEXT[], reasons JSONB,
  ts TIMESTAMPTZ, sim_ts TIMESTAMPTZ);

CREATE TABLE consumer_offsets (consumer TEXT PRIMARY KEY, segment TEXT, "offset" BIGINT,
  high_water_sim_ts TIMESTAMPTZ, updated_at TIMESTAMPTZ);
CREATE TABLE meta (key TEXT PRIMARY KEY, value TEXT);     -- schema_version, ...
```
**Product contract view.** The plan's integration record (`ts, volume, path, user, client_ip, type_class, classifier, ext_mismatch, sha256, tlsh, size`) is a SQL view over `artifacts` per mode. `tlsh` is `null` until v2.

### 6.4 Threat feed file (MalwareBazaar CSV export format)
```
# DwTA fabricated feed (MalwareBazaar export format)
# first_seen_utc,sha256_hash,md5_hash,sha1_hash,reporter,file_name,file_type_guess,mime_type,signature,clamav,vtpercent,imphash,ssdeep,tlsh
"2026-11-06 10:00:00","3f9a…","…","…","dwta-lab","Q3-Report.pdf","exe","application/x-dosexec","DwTA.TestFamily","n/a","n/a","n/a","n/a","n/a"
```

### 6.5 `dwta.incident/v1`
```json
{"schema":"dwta.incident/v1","incident_id":"inc-r-20261103-061240-1","source":"scenario","kind":"ransomware",
 "detected_sim_ts":"2026-11-08T02:05:00Z",
 "user":"svc-backup","client_ip":"10.250.1.66","client_name":"SRV-BKP01","volumes":["finance"],
 "restore_time_sim_ts":"2026-11-02T00:00:00Z",
 "description":"Mass encryption on finance (simulated Ransomware Protection alert)"}
```

### 6.6 Review list file (`run/incidents/<id>/review-list.json`)
```json
{"incident_id":"inc-…","mode":"rules","window":{"from":"2026-11-02T00:00:00Z","to":"2026-11-08T02:05:00Z"},
 "config":{"version":"2026-11-03.2","sha256":"…"},"incomplete":false,
 "items":[
  {"item_id":41,"rank":1,"item_kind":"file","artifact_id":17,"volume":"finance","path":"/Q3/Q3-Report.pdf",
   "sha256":"3f9a…","change_kind":"new_file","version_no":1,"score":230,"suspicion_level":"high","decision":"pending",
   "av":{"verdict":"not_scanned","signature":null,"engine":"none","definitions":null,"scanned_at":null},
   "reasons":[{"signal":"S-FEED","weight":100,"evidence":{"feed_matches":[4]}},
              {"signal":"S-USER","weight":40,"evidence":{"artifacts":[17]}},
              {"signal":"S-IP","weight":20,"evidence":{"artifacts":[17]}},
              {"signal":"S-HOST","weight":10,"evidence":{"artifacts":[17]}},
              {"signal":"S-EXEC-DOC","weight":30,"evidence":{"artifacts":[17]}},
              {"signal":"S-EXT","weight":30,"evidence":{"artifacts":[17]}}]},
  {"item_id":42,"rank":2,"item_kind":"attack_outputs_group","count":41,
   "samples":["/Q1/Budget.xlsx.dwtalocked","/README-DWTA-TEST.txt"],"writer":"svc-backup",
   "from":"2026-11-08T02:00:01Z","to":"2026-11-08T02:00:04Z","suspicion_level":"high","decision":"pending"}]}
```

### 6.7 Restore-exclusion manifest (`run/incidents/<id>/restore-exclusions.json`)
The hand-off from the pre-restore review to the restore step: **restore to `restore_time`, except these files**. Written only by `finalize`.
```json
{"schema":"dwta.restore_exclusions/v1","incident_id":"inc-…","restore_time":"2026-11-02T00:00:00Z",
 "finalized_by":"admin","finalized_at":"2026-11-08T02:30:00Z","mode":"rules",
 "remove":[
  {"item_id":41,"volume":"finance","path":"/Q3/Q3-Report.pdf","file_id":9,"artifact_id":17,"sha256":"3f9a…",
   "change_kind":"new_file","suspicion_level":"high","score":230,"signals":["S-FEED","S-USER","S-IP","S-HOST","S-EXEC-DOC","S-EXT"],
   "decided_by":"admin","decided_at":"2026-11-08T02:20:00Z","action":"quarantine"},
  {"item_id":42,"item_kind":"attack_outputs_group","members":41,"action":"quarantine"}],
 "kept":[],
 "summary":{"listed":2,"remove":2,"keep":0}}
```

All schemas (JSON Schema for 6.1, 6.2, 6.5, 6.6, 6.7; SQL migrations for 6.3; CSV header for 6.4) MUST live in `specs/` and be checked by contract tests before any implementation (plan, H0–H3).

## 7. Acceptance tests (v1)

| ID | Test | Pass criterion | Blocks |
|---|---|---|---|
| AT-01 | Run Cells P1–P2 on the reference Mac | Every check reports PASS or INFO (machine still matches §3) | Env |
| AT-02 | Cold start `docker compose -p <INSTANCE> up -d` **with the network unplugged** | All containers healthy in ≤ 60 s | B2 |
| AT-03 | Each client lists its shares; a wrong password is rejected; `software` is read-only for jdoe and svc-backup | As stated | B1, B2 |
| AT-04 | Mount `finance` from macOS on `SMB_PORT`, copy a file, unmount | File present in `data/finance`; event has `client_origin="host"` | B1, B4 |
| AT-05 | One `put` from each client container | `audit.log` has lines with the correct `%u`, `%I`, `%m`, `%S` for each | B2 |
| AT-06 | Start a second instance (`dwta2`) while `dwta1` runs | Different port and subnet are picked automatically, both pass AT-03, no conflicts | B2, Env |
| AT-07 | `dwta-sim run dwell-window`, then compare the collector output with `groundtruth.jsonl` | **Recall 100 %**, **0 duplicates**, user, IP and client_name match for every record | B3, B4 |
| AT-08 | `dwell-window` legitimate edits (office style, temp-file-then-rename) | Exactly one `write_close` per save, on the final path, `via_rename=true` | B3, B4 |
| AT-09 | Kill the collector mid-scenario and restart it | Still 100 % recall, 0 duplicates (COL-07/08) | B4 |
| AT-10 | `run dwell-window`, then `restore <run_id>` | Day 7 encryption events attributed to svc-backup from `.66`; after restore every SHA-256 matches the corpus pack | B3 |
| AT-11 | Point the simulator at a non-lab host, use an unapproved payload, or target a path outside the seed manifest | `validate` and `run` refuse with a non-zero exit; nothing is written | B3 |
| AT-12 | Copy 1,000 files with the collector running vs stopped | Wall-time difference recorded; no user-visible slowdown | B4 |
| AT-13 | `COLLECTOR_SOURCE=watcher` during `run dwell-window` | Recall 100 % for `write_close`, with `user=null` | B4 |
| AT-14 | Run `dwell-window` and score the collector against its ground truth | 100 % recall, 0 duplicates; `sim_ts` within 1 s of the ground-truth `sim_ts` | B3, B4 |
| AT-15 | `corpus build --profile default --seed 42` twice, in different folders | Identical `manifest.json` and file SHA-256s | B3 |
| AT-16 | Replay the same `audit.log` into an empty journal twice | Byte-identical journal; DuckDB query `SELECT count(*) FROM 'run/journal/**/*.jsonl*'` equals the event count | B4 |
| AT-17 | **Day 1**: after `plant` in `dwell-window` | Artifact `Q3-Report.pdf`: `type_rules = pe_binary`, `ext_mismatch_rules = 1`, SHA-256 equals ground truth; one `write_time` **high** alert (R-W2) | B5, B6 |
| AT-18 | Classifier test corpus (`tests/corpus/disguised/`: PE/ELF/script/LNK renamed to doc names, plus genuine docs) | **Rules catch every disguised case**; Magika equal or better; 0 alerts on genuine docs | B5 |
| AT-19 | Same executable copied to the `software` share | No R-W1/R-W2 alert (false-positive control) | B5 |
| AT-20 | Kill the classifier mid-run, restart; also restart the `ledger` container mid-run | No lost or duplicate artifacts (CLS-05); services reconnect | B5, B6 |
| AT-21 | Stop Magika (bad model path) | Rules mode unaffected; `type_ai = null`; no crash (CLS-06) | B5 |
| AT-22 | A 50 MB PDF is written, then overwritten (RB-AT-01 adapted) | Each version: exactly **one** full read (reason `pass`); hash, type and analyzer results all from it | B5, §5.11 |
| AT-23 | Rename, delete, `publish_feed` with 10,000 hashes, `raise_incident`, `replan` (RB-AT-02/03) | **Zero** content bytes read by any service | B5, B8, B10 |
| AT-24 | A 70 MB file is written (over the hash cap) | ≤ 131,093 bytes read; `sha256 = null`; `analysis = partial` | B5 |
| AT-25 | Full `dwell-window` run (RB-AT-04/07) | `Q3-Report.pdf` has exactly one full read in its life; all reads are of the current file at the evaluated path | B5 |
| AT-26 | `jdoe` writes a real `Budget.pdf` (D1); `svc-backup` overwrites it with the benign PE (D3) | Version 2 linked to version 1 (`change_kind = overwrite`); **R-W6** alert citing both; in the review list the item has `S-TYPE-CHANGE` and `change_kind = overwrite` | B5, B6, B10 |
| AT-27 | The same document re-saved with identical content | New version with `same_content = true`; no new alert | B5 |
| AT-28 | Rename `Q3-Report.pdf` → `Q3-Final.pdf`, then an office-style save onto it | Same `file_id` throughout; current path updated; the save is version 2 (`overwrite`) | B4, B5 |
| AT-29 | **Day 5**: `publish_feed` | Exactly one `feed_match` **critical** alert, for `Q3-Report.pdf`, `sim_ts` = D5 10:00; the 200 decoys match nothing; **no new audit-log lines and no file opens on the share** | B7, B8 |
| AT-30 | Drop the same feed file again | No new entries, matches or alerts | B8 |
| AT-31 | Publish a feed **before** planting a file with that hash | `feed_match` alert raised at write time (reverse path, MAT-04) | B5, B8 |
| AT-32 | **Day 7**: `raise_incident` with restore time D1 00:00 | Review list: `Q3-Report.pdf` is **rank 1** (score 230) in both modes; encrypted copies + note appear as **one** attack-outputs group; **no** legitimate edit is listed; every reason cites an existing ledger row | B9, B10 |
| AT-33 | Restore time set to D2 00:00 (after the plant) | `Q3-Report.pdf` not in the list (documented caveat, OQ-02) | B9, B10 |
| AT-34 | Incident with `user` and `client_ip` omitted | Plan still ranks `Q3-Report.pdf` first on content and feed signals | B10 |
| AT-35 | `dwta-incident raise` without a restore time; then `set-restore-time` with a time after detection; then a valid one | State `awaiting_restore_time` with no review; invalid time rejected with a reason; valid time produces the list | B9, B10 |
| AT-36 | Ad-hoc `plant` of a 2nd payload at D7 02:01 before raising the incident | The 2nd payload is listed as its own item (not folded into the group) | B3, B10 |
| AT-37 | Change `S-EXT` weight in `ranking.toml`, run `dwta-incident replan` | New list reflects the weight, with no code change or restart; it records the new config hash; earlier decisions are kept | B10 |
| AT-38 | `dwta-incident decide <id> <item> remove`, `keep` on another item, then `finalize` | Decisions, admin and time stored; `restore-exclusions.json` lists exactly the removed items; **no file on the share changes** | B10 |
| AT-39 | `finalize` while an item is still `pending`; then change a decision after `finalize` | Both refused with a reason; manifest unchanged | B10 |
| AT-40 | Review list of the `dwell-window` run | `Q3-Report.pdf` level **high** (score 230); levels follow the `[levels]` bands; editing a band in `ranking.toml` changes levels on `replan` | B10 |
| AT-41 | Review list of the `dwell-window` run (v1) | Every file item has `av.verdict = not_scanned`, the group has no AV verdict, and the review is not delayed. v2 adds: the test PE is `detected` on rescan, gains S-AV and stays rank 1 | B10 |
| AT-42 | Run the whole `dwell-window` scenario twice from `reset` | Identical artifacts (except wall-clock fields), alerts and plans | B3–B10 |

**Verification items** (spike first, then pin each answer into a v1.x revision):
- **V-01**: on Debian Bookworm Samba, confirm whether `full_audit:syslog = false` writes to the smbd log, and the exact op names (`openat` vs `open`, `pwrite_send`, `renameat`).
- **V-02**: confirm the source IP that Samba sees for macOS-host connections on Docker Desktop (gateway vs VM IP), and update COL-32.
- **V-03**: confirm Finder and `mount_smbfs` accept a non-445 port on macOS 26 (`smb://host:port/share`).
- **V-04**: `magika` installs and runs offline on `linux/arm64` (`python:3.12-slim`), with the model bundled in the wheel; measure load time and RAM.
- **V-05**: `yara-x` Python wheel is available for `linux/arm64`.
- **V-06**: `postgres:16-alpine` (arm64) starts in ≤ 10 s with the lab's settings and stays ≤ 256 MB RAM under AT-12 load.
- **V-07**: confirm the current MalwareBazaar CSV export columns and header format (one manual download at prep time, hashes only).

## 8. Repository layout, configuration and deferred infrastructure

### 8.1 Layout created by v1
```
hackathon/
  DwTA-POC-Implementation-Plan-v0.2.md
  DwTA-Spec-v1-Foundation.ipynb          # this spec
  dwta-poc/
    specs/                               # JSON schemas, ledger.sql, magic.toml, ext_map.toml, magika_map.toml, ranking.toml, attack_outputs.toml, read_budget.toml, yara/
    lab/
      docker-compose.yml                 # samba, collector, client-* (all parameterized by .env)
      samba/      Dockerfile, smb.conf.tmpl, entrypoint.sh
      smbclient/  Dockerfile
      artifacts/  benign_pe.bin, (eicar.txt, Budget.docm, Payroll.lnk later, after approval)
      corpus/     default-42/ (built once by `dwta-sim corpus build`; git-ignored, reproducible)
      <INSTANCE>/ .env, data/{finance,software}/, logs/, run/, feeds/{inbox,processed,rejected}/, incidents/inbox/   # git-ignored
      (Postgres data lives on the Docker named volume <INSTANCE>_pgdata, LED-01)
    services/collector/                  # tailer, watcher, correlator, sinks
    services/classifier/                 # pipeline, typers/{rules,magika}, analyzers/{oletools,yara}
    services/ledger/                     # migrations/*.sql, roles/grants, repository, export CLI
    services/matcher/                    # feed inbox, parser, matcher
    services/incident/                   # window query, attack-output grouping, ranking, review-list writer, dwta-incident CLI
    tools/dwta_sim/                      # B3 CLI: runner, actions/, scenarios/*.toml
    tests/                               # contract + acceptance tests (AT-xx)
```

### 8.2 Configuration (`lab/<INSTANCE>/.env`)
| Variable | Default | Notes |
|---|---|---|
| `INSTANCE` | `dwta1` | Also the Compose project name (`-p ${INSTANCE}`); prefixes networks and containers |
| `SMB_PORT` | First free port in 14450–14499 | Picked by Cell P2 |
| `LAB_SUBNET` | First free `/24` in `10.250.1–19.0/24` | Picked by Cell P2; client IPs derive from it |
| `SMB_PW_<IDENTITY>` | Random (generated by P4) | Never committed |
| `SIM_KEY`, `SIM_IV` | Random hex (P4) | Simulator encryption |
| `COLLECTOR_SOURCE` | `both` | `audit \| watcher \| both` (D-03) |
| `RENAME_WINDOW` | `5` | Seconds |
| `SIM_EPOCH` | `2026-11-02T09:00:00Z` | Default `sim_epoch` if a scenario doesn't set one |
| `MAGIKA_MIN_SCORE` | `0.8` | Below this, rules_ai falls back to rules typing |
| `LEDGER_PW_<ROLE>` | Random (P4) | Postgres role passwords (LED-03); never committed |
| `LEDGER_HOST_PORT` | unset | Optional; first free port in 15432–15499, bound to `127.0.0.1` (LED-01) |
| `ENGINE_BARRIER_TIMEOUT` | `30` | Seconds to wait for the classifier to catch up (§5.10.1) |

Ranking weights, `min_score` and `max_window_days` are **not** env vars; they live in `specs/ranking.toml` (ENG-07). The window has no setting at all, because the admin supplies the restore time per incident (D-22).

### 8.3 ClamAV container (v2, D-37)
**What it is.** ClamAV is the open-source antivirus engine that Nasuni Antivirus Protection uses. A *ClamAV container* is a Docker image (for example the official `clamav/clamav`) that runs the engine with an offline copy of its signature database (about 300 MB, pulled at prep time).

**Two jobs in v2:**
1. **Day-1 baseline:** scan the planted `Q3-Report.pdf` on simulated Day 1 and report **clean**. That proves the gap is about *timing and context*, not a weak engine.
2. **Step ⑤ rescan:** the incident engine asks it to rescan only the flagged files before the admin sees the list (§5.10.4).

v1 ships the `AvScanner` interface with the `none` implementation (`not_scanned`), so v2 only adds the container and the `clamav` scanner.

## 9. Open questions

| ID | Question | Proposal | Owner | Needed by |
|---|---|---|---|---|
| OQ-01 | Final ranking weights (§5.10.3) | **Proposed values are in use** until the team provides final ones | Team | Before demo script |
| OQ-02 | Safety net for a restore time set too late: also show artifacts **before** the window that have a feed match (S-FEED) or a disguised-executable alert (R-W2), as a separate "known bad, before your restore time" section? | Yes: cheap, and it catches the main way the admin's estimate can be wrong | Security / PM | Before demo script |
| OQ-03 | Product verifications for the Edge team: ClamAV version on the Edge, 1.5+ needed (VERIFY-RB-1); libclamav vs clamd (RB-2); does the AV read files > 25 MB and report container/tail-defined types (RB-3); can a per-snapshot pass get the changed-file list (RB-4) | Ask the Edge AV owners. Affects the product story, not the POC | PM | Before product pitch |

### 9.1 Decision log
| ID | Decision |
|---|---|
| D-01 | Scenario files are **TOML** (stdlib `tomllib`) |
| D-02 | v1 implements **only the `dwell-window` scenario**; the others are a catalog, added on a need basis |
| D-03 | Collector runs **both** sources: the audit tailer is primary, the watcher cross-checks |
| D-04 | Read (`pread`) operations are **not audited** in v1 (log volume); revisit for incident ranking in v3 |
| D-05 | LLM blocks (later) use **Ollama running natively** on the Mac (0.22.0, already installed), not in Docker |
| D-06 | The simulated clock is only a **mapping from real time to story time**, owned by the simulator. Downstream blocks (ledger, incident engine, UI) just read `sim_ts` from events and never touch `simclock.json` |
| D-07 | UNTY-89818 is not used; the simulator is standalone |
| D-08 | The corpus is built **once, outside the simulation loop**, as a deterministic corpus pack. Other corpus packs are added later only if needed (BL-08) |
| D-09 | Collector output uses **three tiers**: raw audit log (Samba), **append-only JSONL event journal owned by the collector**, and a ledger database owned by the ledger block (B6; Postgres per D-20). DuckDB is used for ad hoc queries on the journal (§5.4.6) |
| D-10 | Initial LLM versions use **Ollama (native) with the already-installed `llama3.1`**. Foundation-sec-8B is revisited only after the narrator works |
| D-11 | The harmless macro `.docm` is **deferred to classifier work**. Security eng. creates it once in Word; InfoSec approves it before use |
| D-12 | Attacker story: **v1 uses (A), the look-alike service account `svc-backup`**. Later versions move to **(B), stolen credentials** (attacker logs in as `jdoe` from `SRV-BKP01`), the most common real-world pattern. That needs the `compromised-account` scenario (BL-05) and the per-user baseline (BL-07) |
| D-13 | **No Windows client** for now. Real-client save behavior is emulated with office-style (temp-file-then-rename) writes; AT-08 covers it |
| D-14 | Threat feed in v1 is **fabricated and published by the simulator** (`publish_feed`, Day 5) into a feed inbox. A standalone feed service comes later; **no real MalwareBazaar** support for now |
| D-15 | Classifier reads file bytes **directly from disk** (read-only mount), not over SMB |
| D-16 | Alerts go to the ledger's `alerts` table **and** `run/alerts.jsonl`; API/UI reads them later |
| D-17 | Incident engine produces a ranked **review list only**; executing the admin's decisions comes with API/UI |
| D-18 | Incidents are raised by **both** the scenario (`raise_incident`) and a CLI, standing in for Ransomware Protection |
| D-19 | Classifier v1 = rules core + oletools + YARA-X + Magika. TLSH and entropy/pefile go to v2 |
| D-20 | Ledger runs on **Postgres 16** (`postgres:16-alpine`, arm64), not SQLite. Reason: three writer services in separate containers need real concurrent writes; it also fits a multi-Edge future. Schema kept portable (LED-05) |
| D-21 | Ranking weights, enabled signals, thresholds and classifier rule severities are **configuration files in `specs/`**, re-read per review, with the config hash recorded on every review list |
| D-22 | **One window = [admin's restore time → detection time].** The restore time is the latest moment the admin believes had no attacker presence: Targeted Restore's snapshot and our estimate of the attacker's arrival. Required, no default. It covers the dwell period and the attack itself (second payloads) |
| D-23 | The engine lists **only suspicious files that need human input**, not everything Targeted Restore will restore. The admin decides per item (D-33) |
| D-24 | Assume the product incident carries the **detection time, the initiating user, IP and machine**; the restore time comes from the admin. No start time |
| D-25 | Encrypted copies and ransom notes inside the window collapse into **one attack-outputs group** (patterns in `specs/attack_outputs.toml`) with a single decision |
| D-26 | **Product principle:** DwTA adds no file reads beyond the existing antivirus pass. It harvests SHA-256, type, YARA and macro results from ClamAV (1.5+ callbacks, or a small change to the Edge's AV integration). In the POC, the classifier's single content pass stands in for that AV pass |
| D-27 | **Hash every file ≤ 64 MB** (one full read per version, with oletools and YARA in the same pass). Above 64 MB: head + tail only, no hash |
| D-28 | Evaluation mode **`immediate` only** for now; `per_snapshot` later |
| D-29 | Read accounting, the ClamAV baseline, `/metrics/reads` and the demo panel move to **v2**, together with the ClamAV container (D-37) |
| D-30 | Follow-up writes: FW-K1 head/tail window above 64 MB, FW-H1 full hash ≤ 64 MB, FW-H2 cap 64 MB, FW-C3 **no coalescing** (every close is read) |
| D-31 | **File version tracking** in the ledger: a `files` table + version links on `artifacts`, built by the classifier, metadata only (FW-L1) |
| D-32 | **Type-change rule**: R-W6 write-time alert + S-TYPE-CHANGE ranking signal (+30, configurable) when a file turns into an executable class between versions |
| D-33 | **Core idea: pre-restore review.** The admin gives a restore time; before restoring, DwTA lists the suspicious files written since then; the admin marks each **remove** (keep out of the recovery) or **keep** and finalizes; the restore then runs without the removed files. v1 records decisions and writes the **restore-exclusion manifest**; executing it comes with API/UI |
| D-34 | **Remove = move to a locked quarantine folder**: never delete, always reversible |
| D-35 | Suspicion shown as the **raw score plus a level** from configurable bands in `ranking.toml`: high ≥ 100, medium ≥ 60, low ≥ 30; below 30 not listed |
| D-36 | **Core idea step ⑤: rescan the flagged files with antivirus** (today's signatures) before the admin sees the list. Detection adds **S-AV (+100, configurable)**; every item shows its verdict (detected: name / clean / not scanned). Only listed file items are scanned; the attack-outputs group is skipped |
| D-37 | **ClamAV container comes in v2**, for the Day-1 baseline and the step ⑤ rescan. v1 ships the `AvScanner` interface and marks items `not_scanned` |

### 9.2 Attacker story options (D-12)
| | (A) Look-alike service account (**v1**) | (B) Compromised real account (later versions) |
|---|---|---|
| Story | An attacker uses `svc-backup`, a service account that should never write documents, from machine `SRV-BKP01` (`.66`) | An attacker steals **jdoe's** password and logs in as jdoe, but from their own machine `SRV-BKP01` (`.66`) |
| What's suspicious | The **account** itself: a backup account writing an "executable PDF" onto the finance share | Only the **context**: jdoe normally works from `WS-JDOE01` (`.21`) during business hours |
| What detection needs | The identity alone (rules-only ranking can weight "service account wrote to a document share") | A **per-user baseline** of usual IP/machine/hours (§4.1 option 3, BL-07), which v1 doesn't have |
| Realism | Less common in practice | Very common: most ransomware intrusions use valid stolen credentials |
| Demo risk | Low: works with the rules-only core | Higher: it only works if the baseline exists, but it's the more impressive story |

## 10. Backlog for later spec versions
| ID | Item | Target |
|---|---|---|
| **BL-01** | **Parameterize shares and identities** (count, names, groups, IP octets, NetBIOS names, role = legit/attacker) from one `lab/identities.toml`, instead of the fixed v1 tables | **v2** |
| BL-03 | ClamAV container: Day-1 baseline + step ⑤ rescan scanner (`clamav` AvScanner) | v2 |
| BL-05 | `compromised-account` first (D-12, v2), then other catalog scenarios and their actions (`smash-and-grab`, `slow-drip`, `extension-only`, `wiper`, `staging`, `compromised-account`, `disguised-files`, `normal-week`), each when a test needs it | v2+ |
| BL-06 | Narrator (template + Ollama `llama3.1`), API/UI (pre-restore review screen), and **executing the restore-exclusion manifest**: Targeted Restore + move removed files to a locked quarantine folder | v2+ |
| BL-07 | Behavior-pattern detection (§4.1): window rules first, then the per-identity baseline needed by the stolen-credentials story (D-12) | v2–v3 |
| BL-09 | Standalone mock threat-feed service (MalwareBazaar-like API) feeding the same inbox | v2 |
| BL-10 | TLSH similarity (variant matching), entropy and pefile heuristics; EMBER stretch | v2 |
| BL-11 | Real MalwareBazaar hash-export import (hashes only) | Later |
| BL-12 | Read accounting + ClamAV baseline (rchar and modeled) + `/metrics/reads` + demo panel (RB-A1 to A5, RB-AT-05 comparison workload) | v2 |
| BL-14 | Close coalescing (FW-C3): merge rapid repeated closes of one file into one evaluation | v2+ |
| BL-13 | `per_snapshot` evaluation mode with a simulated snapshot timer (RB-M1/M2, RB-AT-06) | v2+ |
| BL-08 | Additional corpus packs (document-heavy, software-share, 1,000-file, sanitized real sample) for classifier and false-positive evaluation | v2+ |

## 11. Work plan: everything to build for v1

Lanes follow the plan's team roles: **Sec** = security / file-share engineer, **BE** = backend / Python engineer, **AI** = ML / AI engineer, **QA** = PM / QA. "Done when" lists the acceptance tests (§7) that must pass.

### 11.1 Work packages
| WP | Work | Blocks | Depends on | Lane | Done when |
|---|---|---|---|---|---|
| WP-01 | **Spec pack and contract tests:** JSON Schemas (§6), ledger migrations (§6.3), config files (`magic`, `ext_map`, `magika_map`, `ranking`, `attack_outputs`, `read_budget`), first YARA rules, acceptance-test skeletons that fail | all | — | All | Schemas in `specs/`; contract tests run red |
| WP-02 | **Spikes** V-01 to V-07 (§7); pin each answer into a v1.x revision | B2, B5, B6, B7 | — | Sec, AI, BE | Each V-item has a recorded answer |
| WP-03 | **Lab base:** Compose project, Samba image + `smb.conf` + `full_audit`, client image, `.env` and secrets, port and subnet picking, safe artifacts | B1, B2 | WP-01 | Sec | AT-01 to AT-06 |
| WP-04 | **Simulator core:** runner, TOML loader and validator, sim clock, ground truth, safety guards, `corpus build`, actions (`seed`, `edit`, office-style writes, `write`, `plant`, `encrypt`, `drop_note`, `sleep`), `restore`, `reset` | B3 | WP-03 | QA, BE | AT-07, AT-10, AT-11, AT-15 |
| WP-05 | **Collector:** audit tailer, handle correlation, temp-file-then-rename, watcher, `both`-mode de-duplication, journal sink, resume state | B4 | WP-03, V-01 | Sec | AT-05, AT-07 to AT-09, AT-12, AT-13, AT-16 |
| WP-06 | **Ledger:** Postgres container, migrations, roles and grants, export | B6 | WP-01, V-06 | BE | Migrations apply from cold start; roles enforced |
| WP-07 | **Classifier (rules):** journal consumer, `ContentPass` single pass, rules typer, extension and write-time rules R-W1 to R-W6, oletools, YARA-X, version linking, alerts | B5, §5.11 | WP-05, WP-06 | Sec | AT-17 to AT-21 (rules mode), AT-22 to AT-28 |
| WP-08 | **Classifier (AI):** Magika typer, `rules_ai` view, fallback rules | B5 | WP-07, V-04 | AI | AT-18 (Magika ≥ rules), AT-21 |
| WP-09 | **Threat feed and matcher:** `publish_feed` action, inbox, parser, matcher, reverse path | B7, B8 | WP-04, WP-06 | BE | AT-29 to AT-31 |
| WP-10 | **Incident flow:** `raise_incident` action, two-stage CLI, engine (window, barrier, attack-outputs group, ranking with suspicion levels, `AvScanner` hook for the step ⑤ rescan (`none` in v1), pre-restore review list, `remove`/`keep` decisions, `finalize` + restore-exclusion manifest, `replan`) | B9, B10 | WP-07, WP-09 | BE | AT-32 to AT-41 |
| WP-11 | **End-to-end:** `dwell-window` scenario, determinism, cold-start runs | all | WP-04 to WP-10 | QA | AT-14, AT-42 |

### 11.2 Milestones (the plan's gates)
| Gate | Meaning | Work packages |
|---|---|---|
| **Gate 1** | Simulated Day 1 runs end to end from the CLI: copy → audit event → journal → classifier → ledger row → write-time alert | WP-03 to WP-07 |
| **Gate 2** (go/no-go) | The **rules-only** scenario for Days 1, 5 and 7 runs end to end: write-time alert, feed match, pre-restore review list, decisions and restore-exclusion manifest | WP-09, WP-10 |
| **Gate 3** | Both modes (rules and rules + AI) run from a cold start, and every acceptance test in §7 passes | WP-08, WP-11 |

### 11.3 Definition of done for v1
- Every **MUST** requirement in §5 is implemented, and every acceptance test in §7 passes from a cold start on the reference Mac (§3), with the network unplugged.
- Every verification item V-01 to V-07 has a recorded answer.
- Every open question in §9 has an owner and a target date; none blocks the demo story.
- `specs/` holds every contract (§6) and configuration file; no rule, weight or threshold is hard-coded.

## Appendix A — Environment check and setup cells

> Run **P1 → P2 → P3**. P4 changes the machine (pulls images, writes `.env`) and runs only when `RUN_SETUP = True`.

In [ ]:
# Cell P1 — parameters
from pathlib import Path

INSTANCE = "dwta1"                                   # change for a second lab, e.g. "dwta2"
SMB_PORT_RANGE = range(14450, 14500)                 # never 445
LAB_SUBNET_POOL = [f"10.250.{i}.0/24" for i in range(1, 20)]
MIN_PYTHON = (3, 14)                                 # reference Mac: 3.14.0
MIN_DOCKER_MEM_GB = 7                                # reference Mac: ~7.7 GB Docker VM
MIN_FREE_DISK_GB = 10
BASE_IMAGES = ["debian:bookworm-slim", "alpine:3.20", "python:3.12-slim", "postgres:16-alpine"]

# Fixed in v1 (BL-01 parameterizes these in v2)
IDENTITIES = ["admin", "jdoe", "asmith", "svc-backup"]
ACTORS = {                # actor container: (host octet, NetBIOS name, default identity, role)
    "client-admin":  (5,  "WS-ADMIN01",  "admin",      "admin"),
    "client-jdoe":   (21, "WS-JDOE01",   "jdoe",       "legit"),
    "client-asmith": (22, "WS-ASMITH01", "asmith",     "legit"),
    "client-attack": (66, "SRV-BKP01",   "svc-backup", "attacker"),
}
SHARES = {"finance": ["admin", "jdoe", "asmith", "svc-backup"], "software": ["admin", "asmith"]}

RUN_SETUP = False                                    # set True to execute Cell P4
LAB_ROOT = Path.cwd() / "dwta-poc" / "lab"
print(f"Instance {INSTANCE!r}; lab root {LAB_ROOT}")

In [ ]:
# Cell P2 — check the Mac still matches the §3 profile (read-only; picks SMB_PORT and LAB_SUBNET)
import ipaddress, json, platform, shutil, socket, subprocess, sys

results = []
def check(pid, name, status, detail=""):
    results.append((pid, name, status, detail))

def run(cmd, timeout=20):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return p.returncode, p.stdout.strip(), p.stderr.strip()
    except (FileNotFoundError, subprocess.TimeoutExpired) as e:
        return 1, "", str(e)

# ENV-01 macOS / arm64
mac_ver = platform.mac_ver()[0]
check("ENV-01", "macOS on Apple Silicon",
      "PASS" if platform.system() == "Darwin" and platform.machine() == "arm64" else "FAIL",
      f"{platform.system()} {mac_ver} {platform.machine()}")

# ENV-05 Python
check("ENV-05", "Python >= %d.%d" % MIN_PYTHON,
      "PASS" if sys.version_info[:2] >= MIN_PYTHON else "FAIL", platform.python_version())

# ENV-02 Docker daemon + compose
docker_ok = False
if shutil.which("docker"):
    rc, out, err = run(["docker", "info", "--format", "{{json .}}"])
    if rc == 0:
        info = json.loads(out)
        docker_ok = True
        check("ENV-02", "Docker engine running", "PASS",
              f"server {info.get('ServerVersion')} {info.get('OSType')}/{info.get('Architecture')}")
        mem_gb = info.get("MemTotal", 0) / 1024**3
        check("ENV-03", f"Docker memory >= {MIN_DOCKER_MEM_GB} GB",
              "PASS" if mem_gb >= MIN_DOCKER_MEM_GB else "FAIL", f"{mem_gb:.1f} GB, {info.get('NCPU')} CPU")
    else:
        check("ENV-02", "Docker engine running", "FAIL", err[:120] or "docker info failed; start Docker Desktop")
    rc, out, _ = run(["docker", "compose", "version", "--short"])
    check("ENV-02b", "Docker Compose v2+", "PASS" if rc == 0 else "FAIL", out)
else:
    check("ENV-02", "Docker CLI installed", "FAIL", "install Docker Desktop")

# ENV-04 disk
free_gb = shutil.disk_usage(Path.home()).free / 1024**3
check("ENV-04", f"Free disk >= {MIN_FREE_DISK_GB} GB",
      "PASS" if free_gb >= MIN_FREE_DISK_GB else "FAIL", f"{free_gb:.0f} GB free")

# ENV-06 native SMB client tools
for tool in ("mount_smbfs", "smbutil"):
    path = shutil.which(tool) or (f"/sbin/{tool}" if Path(f"/sbin/{tool}").exists() else None)
    check("ENV-06", f"{tool} present", "PASS" if path else "FAIL", path or "missing")

# Informational: native Ollama for later LLM blocks (D-05)
if shutil.which("ollama"):
    rc, out, err = run(["ollama", "list"])
    models = [l.split()[0] for l in out.splitlines()[1:]] if rc == 0 else []
    check("INFO", "Ollama (native)", "INFO",
          f"running; models: {', '.join(models) or 'none'}" if rc == 0
          else "installed, not running (ollama serve / brew services start ollama)")
else:
    check("INFO", "Ollama (native)", "INFO", "not installed; only needed by later LLM blocks")

# Informational: macOS SMB server on 445
def port_listening(port, host="127.0.0.1"):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0
check("INFO", "Port 445 on host", "INFO",
      "in use (macOS File Sharing or another server); lab will not use it" if port_listening(445) else "free; lab still won't use it")

# ENV-07 pick a free SMB port (not bound locally, not published by any container)
published = set()
if docker_ok:
    _, out, _ = run(["docker", "ps", "--format", "{{.Ports}}"])
    for token in out.replace(",", " ").split():
        if "->" in token:
            hostpart = token.split("->")[0]
            port = hostpart.rsplit(":", 1)[-1]
            if port.isdigit():
                published.add(int(port))

def port_free(port):
    if port in published or port_listening(port):
        return False
    with socket.socket() as s:
        try:
            s.bind(("127.0.0.1", port)); return True
        except OSError:
            return False

SMB_PORT = next((p for p in SMB_PORT_RANGE if port_free(p)), None)
check("ENV-07", "Free SMB host port", "PASS" if SMB_PORT else "FAIL",
      f"{SMB_PORT} (range {SMB_PORT_RANGE.start}-{SMB_PORT_RANGE.stop - 1})")

# ENV-08 pick a lab subnet that overlaps no Docker network
used_nets = []
if docker_ok:
    _, ids, _ = run(["docker", "network", "ls", "-q"])
    if ids:
        _, out, _ = run(["docker", "network", "inspect", *ids.split(),
                         "--format", "{{range .IPAM.Config}}{{.Subnet}} {{end}}"])
        for cidr in out.split():
            try: used_nets.append(ipaddress.ip_network(cidr, strict=False))
            except ValueError: pass
LAB_SUBNET = next((c for c in LAB_SUBNET_POOL
                   if not any(ipaddress.ip_network(c).overlaps(u) for u in used_nets if u.version == 4)), None)
check("ENV-08", "Free lab subnet", "PASS" if LAB_SUBNET else "FAIL",
      f"{LAB_SUBNET} ({len(used_nets)} existing docker subnets avoided)")

# ENV-09 base images present locally (informational until P4 is run)
if docker_ok:
    missing = [img for img in BASE_IMAGES if run(["docker", "image", "inspect", img])[0] != 0]
    check("ENV-09", "Base images pre-pulled", "PASS" if not missing else "INFO",
          "all present" if not missing else f"missing: {', '.join(missing)}; run P4")

width = max(len(r[1]) for r in results)
for pid, name, status, detail in results:
    print(f"{pid:6} {status:5} {name:<{width}}  {detail}")
failed = [r for r in results if r[2] == "FAIL"]
print("\nRESULT:", "MAC MATCHES REFERENCE PROFILE" if not failed else f"{len(failed)} FAILED")

In [ ]:
# Cell P3 — derived address plan for this instance
net = ipaddress.ip_network(LAB_SUBNET)
hosts = {"samba": 10, "ledger": 20, "collector": 30, "classifier": 31, "matcher": 32, "incident": 33,
         **{k: v[0] for k, v in ACTORS.items()}}
print(f"Compose project : {INSTANCE}")
print(f"Lab subnet      : {net}  (gateway {net.network_address + 1})")
print(f"SMB endpoint    : smb://<identity>@127.0.0.1:{SMB_PORT}/<share>\n")
for name, octet in hosts.items():
    extra = f"{ACTORS[name][1]:12} as {ACTORS[name][2]:11} {ACTORS[name][3]}" if name in ACTORS else ""
    print(f"  {name:18} {str(net.network_address + octet):15} {extra}")
print()
for share, users in SHARES.items():
    print(f"  share {share:9} write: {', '.join(users)}")

In [ ]:
# Cell P4 — setup (only when RUN_SETUP = True): pre-pull images, write lab/<INSTANCE>/.env
import secrets

if not RUN_SETUP:
    print("RUN_SETUP is False; skipping. Set it in Cell P1 to pull images and write .env.")
else:
    for img in BASE_IMAGES:
        print("pulling", img, "...", flush=True)
        rc, _, err = run(["docker", "pull", "--platform", "linux/arm64", img], timeout=900)
        print("  ok" if rc == 0 else f"  FAILED: {err[:200]}")

    inst_dir = LAB_ROOT / INSTANCE
    for sub in ("data/finance", "data/software", "logs", "run"):
        (inst_dir / sub).mkdir(parents=True, exist_ok=True)
    env_file = inst_dir / ".env"
    if env_file.exists():
        print(f"{env_file} exists; not overwriting (delete it to regenerate secrets).")
    else:
        lines = [f"INSTANCE={INSTANCE}", f"SMB_PORT={SMB_PORT}", f"LAB_SUBNET={LAB_SUBNET}",
                 "COLLECTOR_SOURCE=both", "RENAME_WINDOW=5", "SIM_EPOCH=2026-11-02T09:00:00Z",
                 f"SIM_KEY={secrets.token_hex(32)}", f"SIM_IV={secrets.token_hex(16)}"]
        lines += [f"SMB_PW_{k.upper().replace('-', '_')}={secrets.token_urlsafe(18)}" for k in IDENTITIES]
        lines += [f"LEDGER_PW_{r}={secrets.token_urlsafe(18)}" for r in ("ADMIN", "CLASSIFIER", "MATCHER", "INCIDENT", "READER")]
        env_file.write_text("\n".join(lines) + "\n")
        env_file.chmod(0o600)
        (LAB_ROOT / ".gitignore").write_text("*/\n!artifacts/\n!samba/\n!smbclient/\n")
        print(f"wrote {env_file} (mode 600, git-ignored)")